In [ ]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCY BACKEND
# ============================================================

%pip install -q \
    pytorch-crf==0.7.2 \
    pandas


# ------------------------------------------------------------
# Test dependency
# ------------------------------------------------------------

import torch
import pandas as pd
from torchcrf import CRF


print("=" * 70)
print("DEPENDENCY BACKEND")
print("=" * 70)

print("✅ PyTorch :", torch.__version__)
print("✅ Pandas  :", pd.__version__)
print("✅ torchcrf berhasil di-import.")

if torch.cuda.is_available():

    print(
        "✅ GPU aktif :",
        torch.cuda.get_device_name(0)
    )

else:

    print(
        "ℹ️ GPU tidak aktif. Backend tetap dapat menggunakan CPU."
    )


print("\n✅ Cell 1 selesai.")

In [ ]:
# ============================================================
# CELL 2 — MOUNT DRIVE + VALIDASI PATH BACKEND
# ============================================================

from google.colab import drive
from pathlib import Path
import os


# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

MOUNT_ROOT = Path("/content/gdrive")

if not (MOUNT_ROOT / "MyDrive").exists():

    drive.mount(
        str(MOUNT_ROOT)
    )

else:

    print("✅ Google Drive sudah ter-mount.")


# ------------------------------------------------------------
# 2. Project path
# ------------------------------------------------------------

PROJECT_ROOT = (
    MOUNT_ROOT
    / "MyDrive"
    / "BiLSTM-CRF"
)

STREAMLIT_DIR = (
    PROJECT_ROOT
    / "streamlit_app"
)

ASSETS_DIR = (
    STREAMLIT_DIR
    / "assets"
)


# ------------------------------------------------------------
# 3. Model dan mapping
# ------------------------------------------------------------

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "finetuned"
    / "best_finetuned_model.pt"
)

CONFIG_PATH = (
    PROJECT_ROOT
    / "models"
    / "finetuned"
    / "finetuning_config.json"
)

CHAR2ID_PATH = (
    PROJECT_ROOT
    / "mappings"
    / "char2id.json"
)

LABEL2ID_PATH = (
    PROJECT_ROOT
    / "mappings"
    / "label2id.json"
)

ID2LABEL_PATH = (
    PROJECT_ROOT
    / "mappings"
    / "id2label.json"
)

METRICS_PATH = (
    PROJECT_ROOT
    / "evaluation"
    / "comparison"
    / "metrics_comparison.json"
)


# ------------------------------------------------------------
# 4. Buat folder backend jika belum ada
# ------------------------------------------------------------

STREAMLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ASSETS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 5. Environment variable
# ------------------------------------------------------------

os.environ[
    "BILSTM_CRF_PROJECT_ROOT"
] = str(PROJECT_ROOT)


# ------------------------------------------------------------
# 6. Validasi file penting
# ------------------------------------------------------------

required_files = {
    "Project root":
        PROJECT_ROOT,

    "Model fine-tuned":
        MODEL_PATH,

    "Config fine-tuning":
        CONFIG_PATH,

    "char2id":
        CHAR2ID_PATH,

    "label2id":
        LABEL2ID_PATH,

    "id2label":
        ID2LABEL_PATH,

    "Metrics evaluation":
        METRICS_PATH,
}


print("=" * 70)
print("VALIDASI PATH BACKEND")
print("=" * 70)

all_ok = True

for name, path in required_files.items():

    exists = path.exists()

    status = (
        "✅ ADA"
        if exists
        else "❌ TIDAK ADA"
    )

    print(
        f"{name:<22}: {status}"
    )

    if not exists:
        print(
            " " * 24,
            path
        )

        all_ok = False


print("\nProject:")
print(PROJECT_ROOT)

print("\nFolder backend:")
print(STREAMLIT_DIR)


# ------------------------------------------------------------
# 7. Final check
# ------------------------------------------------------------

assert all_ok, (
    "❌ Ada file backend penting yang tidak ditemukan. "
    "Jangan lanjut ke Cell 3."
)

print("\n✅ Semua file backend ditemukan.")
print("✅ Cell 2 selesai.")

In [ ]:
# ============================================================
# CELL 3 — LOAD & AUDIT CONFIG + MAPPING
# ============================================================

import json


# ------------------------------------------------------------
# 1. Helper baca JSON
# ------------------------------------------------------------

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(f)


# ------------------------------------------------------------
# 2. Load file
# ------------------------------------------------------------

config = load_json(CONFIG_PATH)

char2id = load_json(CHAR2ID_PATH)
label2id = load_json(LABEL2ID_PATH)
id2label_raw = load_json(ID2LABEL_PATH)

metrics_data = load_json(METRICS_PATH)


# ------------------------------------------------------------
# 3. Normalisasi id2label
#
# JSON menyimpan key sebagai string.
# Kita ubah menjadi integer.
# ------------------------------------------------------------

id2label = {
    int(key): value
    for key, value
    in id2label_raw.items()
}


# ------------------------------------------------------------
# 4. Audit ukuran mapping
# ------------------------------------------------------------

VOCAB_SIZE = len(char2id)
NUM_LABELS = len(label2id)


print("=" * 70)
print("AUDIT CONFIG & MAPPING")
print("=" * 70)

print(
    f"Vocabulary size : {VOCAB_SIZE}"
)

print(
    f"Jumlah label    : {NUM_LABELS}"
)


# ------------------------------------------------------------
# 5. Token wajib
# ------------------------------------------------------------

print("\nTOKEN KHUSUS")
print("-" * 70)

for token in [
    "<PAD>",
    "<UNK>",
]:

    if token in char2id:

        print(
            f"✅ {token:<8}: ID {char2id[token]}"
        )

    else:

        print(
            f"❌ {token} tidak ditemukan."
        )


# ------------------------------------------------------------
# 6. Audit ID char2id
# ------------------------------------------------------------

char_ids = list(
    char2id.values()
)

assert len(char_ids) == len(set(char_ids)), (
    "❌ Ada ID karakter duplikat."
)

assert set(char_ids) == set(
    range(VOCAB_SIZE)
), (
    "❌ ID karakter tidak contiguous dari 0 sampai vocab_size-1."
)

print(
    "\n✅ char2id unik dan contiguous."
)


# ------------------------------------------------------------
# 7. Audit label2id
# ------------------------------------------------------------

label_ids = list(
    label2id.values()
)

assert len(label_ids) == len(set(label_ids)), (
    "❌ Ada ID label duplikat."
)

assert set(label_ids) == set(
    range(NUM_LABELS)
), (
    "❌ ID label tidak contiguous."
)

print(
    "✅ label2id unik dan contiguous."
)


# ------------------------------------------------------------
# 8. Konsistensi label2id ↔ id2label
# ------------------------------------------------------------

mapping_ok = True

for label, idx in label2id.items():

    reverse_label = id2label.get(
        idx
    )

    if reverse_label != label:

        print(
            "❌ Mapping tidak cocok:",
            label,
            idx,
            reverse_label,
        )

        mapping_ok = False


assert mapping_ok, (
    "❌ label2id dan id2label tidak konsisten."
)

print(
    "✅ label2id dan id2label konsisten."
)


# ------------------------------------------------------------
# 9. Tampilkan semua label
# ------------------------------------------------------------

print("\nLABEL HARAKAT")
print("-" * 70)

for idx in sorted(id2label):

    print(
        f"{idx:>2} -> {id2label[idx]}"
    )


# ------------------------------------------------------------
# 10. Tampilkan struktur config
# ------------------------------------------------------------

print("\nCONFIG TOP-LEVEL")
print("-" * 70)

for key, value in config.items():

    if isinstance(
        value,
        (dict, list)
    ):

        print(
            f"{key:<28}: {type(value).__name__}"
        )

    else:

        print(
            f"{key:<28}: {value}"
        )


# ------------------------------------------------------------
# 11. Struktur metrics
# ------------------------------------------------------------

print("\nMETRICS TOP-LEVEL")
print("-" * 70)

print(
    list(metrics_data.keys())
)


# ------------------------------------------------------------
# 12. Final validation
# ------------------------------------------------------------

assert "<PAD>" in char2id, (
    "❌ <PAD> tidak ditemukan."
)

assert "<UNK>" in char2id, (
    "❌ <UNK> tidak ditemukan."
)

assert VOCAB_SIZE > 0
assert NUM_LABELS > 0


print("\n" + "=" * 70)
print("✅ Mapping valid.")
print("✅ Config berhasil dibaca.")
print("✅ Metrics berhasil dibaca.")
print("✅ Cell 3 selesai.")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 4 — AUDIT CHECKPOINT FINE-TUNED [FIXED]
# ============================================================

import torch


print("=" * 70)
print("AUDIT CHECKPOINT FINE-TUNED")
print("=" * 70)


# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device backend :",
    DEVICE
)


# ------------------------------------------------------------
# 2. Load checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)


assert isinstance(
    checkpoint,
    dict
), (
    "❌ Checkpoint bukan dictionary."
)


print("\nTIPE CHECKPOINT")
print("-" * 70)

print(
    "Type:",
    type(checkpoint).__name__
)

print(
    "Top-level keys:"
)

for key in checkpoint.keys():

    print(
        " -",
        key
    )


# ------------------------------------------------------------
# 3. Ambil model_state_dict
# ------------------------------------------------------------

assert "model_state_dict" in checkpoint, (
    "❌ model_state_dict tidak ditemukan."
)

state_dict = checkpoint[
    "model_state_dict"
]


print(
    "\nState dict source : model_state_dict"
)

print(
    "Jumlah parameter tensor:",
    len(state_dict)
)


# ------------------------------------------------------------
# 4. Bersihkan prefix DataParallel jika ada
# ------------------------------------------------------------

clean_state_dict = {}

for key, value in state_dict.items():

    clean_key = key

    if clean_key.startswith(
        "module."
    ):

        clean_key = clean_key[
            len("module.") :
        ]

    clean_state_dict[
        clean_key
    ] = value


state_dict = clean_state_dict


# ------------------------------------------------------------
# 5. Tampilkan struktur tensor
# ------------------------------------------------------------

print("\nSTRUKTUR PARAMETER MODEL")
print("-" * 70)

for key, tensor in state_dict.items():

    print(
        f"{key:<42}"
        f"{tuple(tensor.shape)}"
    )


# ------------------------------------------------------------
# 6. Key wajib
# ------------------------------------------------------------

required_state_keys = [

    "embedding.weight",

    "bilstm.weight_ih_l0",
    "bilstm.weight_hh_l0",
    "bilstm.bias_ih_l0",
    "bilstm.bias_hh_l0",

    "bilstm.weight_ih_l0_reverse",
    "bilstm.weight_hh_l0_reverse",
    "bilstm.bias_ih_l0_reverse",
    "bilstm.bias_hh_l0_reverse",

    "classifier.weight",
    "classifier.bias",

    "crf.start_transitions",
    "crf.end_transitions",
    "crf.transitions",
]


print("\nVALIDASI KEY MODEL")
print("-" * 70)

for key in required_state_keys:

    assert key in state_dict, (
        f"❌ Parameter tidak ditemukan: {key}"
    )

    print(
        f"✅ {key}"
    )


# ------------------------------------------------------------
# 7. Audit embedding
# ------------------------------------------------------------

embedding_shape = tuple(
    state_dict[
        "embedding.weight"
    ].shape
)


print("\nVALIDASI DIMENSI")
print("-" * 70)

print(
    "Embedding        :",
    embedding_shape
)


assert embedding_shape == (
    VOCAB_SIZE,
    int(
        config[
            "embedding_dimension"
        ]
    ),
), (
    "❌ Shape embedding tidak cocok."
)


# ------------------------------------------------------------
# 8. Deteksi hidden size asli dari checkpoint
# ------------------------------------------------------------

bilstm_ih_shape = tuple(
    state_dict[
        "bilstm.weight_ih_l0"
    ].shape
)


bilstm_hh_shape = tuple(
    state_dict[
        "bilstm.weight_hh_l0"
    ].shape
)


print(
    "BiLSTM weight_ih:",
    bilstm_ih_shape
)

print(
    "BiLSTM weight_hh:",
    bilstm_hh_shape
)


# PyTorch LSTM:
#
# weight_ih_l0 =
# (4 * hidden_size, input_size)
#
# Maka hidden_size dapat dihitung langsung.

assert (
    bilstm_ih_shape[0] % 4
    == 0
), (
    "❌ Dimensi gate BiLSTM tidak valid."
)


LSTM_HIDDEN_SIZE = (
    bilstm_ih_shape[0]
    // 4
)


print(
    "Hidden per arah :",
    LSTM_HIDDEN_SIZE
)


# ------------------------------------------------------------
# 9. Validasi input size LSTM
# ------------------------------------------------------------

assert bilstm_ih_shape[1] == int(
    config[
        "embedding_dimension"
    ]
), (
    "❌ Input size BiLSTM tidak sama "
    "dengan embedding dimension."
)


assert bilstm_hh_shape == (
    4 * LSTM_HIDDEN_SIZE,
    LSTM_HIDDEN_SIZE,
), (
    "❌ Shape recurrent BiLSTM tidak valid."
)


# ------------------------------------------------------------
# 10. Reverse direction
# ------------------------------------------------------------

reverse_ih_shape = tuple(
    state_dict[
        "bilstm.weight_ih_l0_reverse"
    ].shape
)


reverse_hh_shape = tuple(
    state_dict[
        "bilstm.weight_hh_l0_reverse"
    ].shape
)


assert reverse_ih_shape == bilstm_ih_shape, (
    "❌ Forward/reverse weight_ih berbeda."
)

assert reverse_hh_shape == bilstm_hh_shape, (
    "❌ Forward/reverse weight_hh berbeda."
)


print(
    "✅ Forward dan backward BiLSTM konsisten."
)


# ------------------------------------------------------------
# 11. Output dimension BiLSTM
# ------------------------------------------------------------

BILSTM_OUTPUT_DIM = (
    LSTM_HIDDEN_SIZE
    * 2
)


print(
    "Output BiLSTM   :",
    BILSTM_OUTPUT_DIM
)


# ------------------------------------------------------------
# 12. Interpretasi hidden_dimension config
# ------------------------------------------------------------

CONFIG_HIDDEN_DIM = int(
    config[
        "hidden_dimension"
    ]
)


print(
    "Config hidden   :",
    CONFIG_HIDDEN_DIM
)


assert CONFIG_HIDDEN_DIM == BILSTM_OUTPUT_DIM, (
    "❌ hidden_dimension pada config tidak "
    "cocok dengan output BiLSTM checkpoint."
)


print(
    "✅ hidden_dimension=256 merepresentasikan "
    "output gabungan BiLSTM."
)


# ------------------------------------------------------------
# 13. Classifier
# ------------------------------------------------------------

classifier_shape = tuple(
    state_dict[
        "classifier.weight"
    ].shape
)


print(
    "Classifier      :",
    classifier_shape
)


assert classifier_shape == (
    NUM_LABELS,
    BILSTM_OUTPUT_DIM,
), (
    "❌ Shape classifier tidak cocok."
)


assert tuple(
    state_dict[
        "classifier.bias"
    ].shape
) == (
    NUM_LABELS,
), (
    "❌ Bias classifier tidak cocok."
)


print(
    "✅ Classifier cocok."
)


# ------------------------------------------------------------
# 14. CRF
# ------------------------------------------------------------

print("\nVALIDASI CRF")
print("-" * 70)


assert tuple(
    state_dict[
        "crf.start_transitions"
    ].shape
) == (
    NUM_LABELS,
)

assert tuple(
    state_dict[
        "crf.end_transitions"
    ].shape
) == (
    NUM_LABELS,
)

assert tuple(
    state_dict[
        "crf.transitions"
    ].shape
) == (
    NUM_LABELS,
    NUM_LABELS,
)


print(
    "✅ start_transitions :",
    tuple(
        state_dict[
            "crf.start_transitions"
        ].shape
    )
)

print(
    "✅ end_transitions   :",
    tuple(
        state_dict[
            "crf.end_transitions"
        ].shape
    )
)

print(
    "✅ transitions       :",
    tuple(
        state_dict[
            "crf.transitions"
        ].shape
    )
)


# ------------------------------------------------------------
# 15. Epoch checkpoint
# ------------------------------------------------------------

CHECKPOINT_EPOCH = checkpoint.get(
    "epoch"
)


print("\nMETADATA CHECKPOINT")
print("-" * 70)

print(
    "Epoch checkpoint :",
    CHECKPOINT_EPOCH
)

print(
    "Best epoch config:",
    config[
        "best_epoch"
    ]
)


assert int(
    CHECKPOINT_EPOCH
) == int(
    config[
        "best_epoch"
    ]
), (
    "❌ Epoch checkpoint tidak sama "
    "dengan best_epoch config."
)


print(
    "✅ Epoch checkpoint cocok."
)


# ------------------------------------------------------------
# 16. Mapping checkpoint vs mapping final
# ------------------------------------------------------------

if "char2id" in checkpoint:

    assert (
        checkpoint["char2id"]
        ==
        char2id
    ), (
        "❌ char2id checkpoint berbeda."
    )

    print(
        "✅ char2id checkpoint cocok."
    )


if "label2id" in checkpoint:

    assert (
        checkpoint["label2id"]
        ==
        label2id
    ), (
        "❌ label2id checkpoint berbeda."
    )

    print(
        "✅ label2id checkpoint cocok."
    )


# ------------------------------------------------------------
# 17. Hitung total parameter dari checkpoint
# ------------------------------------------------------------

TOTAL_PARAMETERS = sum(
    tensor.numel()
    for tensor in state_dict.values()
)


print("\nTOTAL PARAMETER")
print("-" * 70)

print(
    f"{TOTAL_PARAMETERS:,}"
)


# ------------------------------------------------------------
# 18. Konstanta final backend
# ------------------------------------------------------------

EMBEDDING_DIM = int(
    config[
        "embedding_dimension"
    ]
)

NUM_LAYERS = int(
    config[
        "number_of_lstm_layers"
    ]
)

DROPOUT = float(
    config[
        "dropout"
    ]
)

MAX_SEQUENCE_LENGTH = int(
    config[
        "max_sequence_length"
    ]
)

PAD_ID = int(
    char2id[
        "<PAD>"
    ]
)

UNK_ID = int(
    char2id[
        "<UNK>"
    ]
)


# ------------------------------------------------------------
# 19. Ringkasan arsitektur
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ARSITEKTUR MODEL TERDETEKSI")
print("=" * 70)

print(
    "Vocabulary           :",
    VOCAB_SIZE
)

print(
    "Jumlah label         :",
    NUM_LABELS
)

print(
    "Embedding dimension  :",
    EMBEDDING_DIM
)

print(
    "Hidden per direction :",
    LSTM_HIDDEN_SIZE
)

print(
    "BiLSTM output dim    :",
    BILSTM_OUTPUT_DIM
)

print(
    "Config hidden dim    :",
    CONFIG_HIDDEN_DIM
)

print(
    "LSTM layers          :",
    NUM_LAYERS
)

print(
    "Dropout              :",
    DROPOUT
)

print(
    "Max sequence         :",
    MAX_SEQUENCE_LENGTH
)

print(
    "PAD ID               :",
    PAD_ID
)

print(
    "UNK ID               :",
    UNK_ID
)

print(
    "Total parameters     :",
    f"{TOTAL_PARAMETERS:,}"
)


print("\n✅ Checkpoint valid.")
print("✅ Arsitektur berhasil dideteksi dari bobot asli.")
print("✅ Tidak ada mismatch model.")
print("✅ Cell 4 selesai.")

In [ ]:
# ============================================================
# CELL 5 — BUILD MODEL + LOAD CHECKPOINT
# ============================================================

import torch
import torch.nn as nn

from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
)

from torchcrf import CRF


print("=" * 70)
print("BUILD BiLSTM-CRF")
print("=" * 70)


# ------------------------------------------------------------
# 1. Definisi model
# ------------------------------------------------------------

class BiLSTMCRF(nn.Module):

    def __init__(
        self,
        vocab_size,
        num_labels,
        embedding_dim,
        hidden_dim,
        num_layers,
        dropout,
        pad_idx,
    ):

        super().__init__()


        # ----------------------------------------------------
        # Validasi hidden dimension
        #
        # hidden_dim = output gabungan BiLSTM
        # sehingga hidden per arah = hidden_dim // 2
        # ----------------------------------------------------

        assert hidden_dim % 2 == 0, (
            "hidden_dim harus genap untuk BiLSTM."
        )

        self.hidden_per_direction = (
            hidden_dim // 2
        )


        # ----------------------------------------------------
        # Embedding
        # ----------------------------------------------------

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx,
        )


        self.embedding_dropout = nn.Dropout(
            dropout
        )


        # ----------------------------------------------------
        # BiLSTM
        # ----------------------------------------------------

        self.bilstm = nn.LSTM(
            input_size=embedding_dim,

            hidden_size=
                self.hidden_per_direction,

            num_layers=num_layers,

            batch_first=True,

            bidirectional=True,

            dropout=(
                dropout
                if num_layers > 1
                else 0.0
            ),
        )


        # ----------------------------------------------------
        # Dropout setelah BiLSTM
        # ----------------------------------------------------

        self.output_dropout = nn.Dropout(
            dropout
        )


        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------

        self.classifier = nn.Linear(
            hidden_dim,
            num_labels,
        )


        # ----------------------------------------------------
        # CRF
        # ----------------------------------------------------

        self.crf = CRF(
            num_tags=num_labels,
            batch_first=True,
        )


    # --------------------------------------------------------
    # Forward → emission scores
    # --------------------------------------------------------

    def forward(
        self,
        input_ids,
        lengths,
    ):

        embeddings = self.embedding(
            input_ids
        )

        embeddings = self.embedding_dropout(
            embeddings
        )


        # ----------------------------------------------------
        # Packed sequence
        # ----------------------------------------------------

        packed = pack_padded_sequence(
            embeddings,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )


        packed_output, _ = self.bilstm(
            packed
        )


        bilstm_output, _ = pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=input_ids.size(1),
        )


        bilstm_output = self.output_dropout(
            bilstm_output
        )


        emissions = self.classifier(
            bilstm_output
        )


        return emissions


    # --------------------------------------------------------
    # CRF Loss
    # --------------------------------------------------------

    def loss(
        self,
        input_ids,
        tags,
        mask,
        lengths,
    ):

        emissions = self.forward(
            input_ids,
            lengths,
        )


        log_likelihood = self.crf(
            emissions,
            tags,
            mask=mask,
            reduction="token_mean",
        )


        return -log_likelihood


    # --------------------------------------------------------
    # Viterbi Decode
    # --------------------------------------------------------

    def decode(
        self,
        input_ids,
        mask,
        lengths,
    ):

        emissions = self.forward(
            input_ids,
            lengths,
        )


        return self.crf.decode(
            emissions,
            mask=mask,
        )


# ------------------------------------------------------------
# 2. Instantiate model
# ------------------------------------------------------------

model = BiLSTMCRF(

    vocab_size=
        VOCAB_SIZE,

    num_labels=
        NUM_LABELS,

    embedding_dim=
        EMBEDDING_DIM,

    hidden_dim=
        CONFIG_HIDDEN_DIM,

    num_layers=
        NUM_LAYERS,

    dropout=
        DROPOUT,

    pad_idx=
        PAD_ID,
)


model = model.to(
    DEVICE
)


# ------------------------------------------------------------
# 3. Load checkpoint secara STRICT
# ------------------------------------------------------------

load_result = model.load_state_dict(
    state_dict,
    strict=True,
)


print("\nLOAD CHECKPOINT")
print("-" * 70)

print(
    "Missing keys   :",
    load_result.missing_keys
)

print(
    "Unexpected keys:",
    load_result.unexpected_keys
)


assert not load_result.missing_keys, (
    "❌ Ada parameter model yang tidak ditemukan."
)

assert not load_result.unexpected_keys, (
    "❌ Ada parameter checkpoint yang tidak digunakan."
)


# ------------------------------------------------------------
# 4. Evaluation mode
# ------------------------------------------------------------

model.eval()


print(
    "✅ Checkpoint berhasil dimuat secara strict."
)

print(
    "✅ Model masuk evaluation mode."
)


# ------------------------------------------------------------
# 5. Hitung parameter
# ------------------------------------------------------------

model_total_parameters = sum(

    parameter.numel()

    for parameter
    in model.parameters()
)


trainable_parameters = sum(

    parameter.numel()

    for parameter
    in model.parameters()

    if parameter.requires_grad
)


print("\nPARAMETER MODEL")
print("-" * 70)

print(
    "Total parameter    :",
    f"{model_total_parameters:,}"
)

print(
    "Trainable parameter:",
    f"{trainable_parameters:,}"
)


assert (
    model_total_parameters
    ==
    TOTAL_PARAMETERS
), (
    "❌ Total parameter model berbeda "
    "dengan checkpoint."
)


print(
    "✅ Total parameter cocok dengan checkpoint."
)


# ------------------------------------------------------------
# 6. Test satu sequence Arab gundul
# ------------------------------------------------------------

TEST_TEXT = (
    "قال الشيخ الإمام العالم العلامة شمس الدين"
)


# Encode karakter
test_ids = [

    char2id.get(
        char,
        UNK_ID,
    )

    for char
    in TEST_TEXT
]


assert len(test_ids) > 0


# Tensor [batch, sequence]
test_tensor = torch.tensor(
    [test_ids],
    dtype=torch.long,
    device=DEVICE,
)


# Panjang sequence
test_lengths = torch.tensor(
    [len(test_ids)],
    dtype=torch.long,
)


# Mask
test_mask = (
    test_tensor
    != PAD_ID
)


# ------------------------------------------------------------
# 7. Forward test
# ------------------------------------------------------------

with torch.no_grad():

    emissions = model(
        test_tensor,
        test_lengths,
    )

    decoded = model.decode(
        test_tensor,
        test_mask,
        test_lengths,
    )


print("\nTEST FORWARD")
print("-" * 70)

print(
    "Input:",
    TEST_TEXT
)

print(
    "Panjang input:",
    len(TEST_TEXT)
)

print(
    "Emission shape:",
    tuple(
        emissions.shape
    )
)

print(
    "Jumlah sequence hasil:",
    len(decoded)
)

print(
    "Panjang label hasil:",
    len(
        decoded[0]
    )
)


# ------------------------------------------------------------
# 8. Validasi hasil decode
# ------------------------------------------------------------

assert emissions.shape == (
    1,
    len(TEST_TEXT),
    NUM_LABELS,
), (
    "❌ Shape emission tidak sesuai."
)


assert len(decoded) == 1, (
    "❌ Jumlah hasil decode salah."
)


assert len(
    decoded[0]
) == len(
    TEST_TEXT
), (
    "❌ Panjang label prediksi tidak sama "
    "dengan panjang karakter input."
)


assert all(
    0 <= label_id < NUM_LABELS
    for label_id
    in decoded[0]
), (
    "❌ Decode menghasilkan ID label invalid."
)


# ------------------------------------------------------------
# 9. Preview label prediksi
# ------------------------------------------------------------

predicted_labels = [

    id2label[
        label_id
    ]

    for label_id
    in decoded[0]
]


print("\nPREVIEW LABEL")
print("-" * 70)

for char, label in zip(
    TEST_TEXT[:20],
    predicted_labels[:20],
):

    print(
        repr(char),
        "->",
        repr(label)
    )


# ------------------------------------------------------------
# 10. Final
# ------------------------------------------------------------

print("\n" + "=" * 70)

print(
    "✅ Model BiLSTM-CRF berhasil dibangun."
)

print(
    "✅ Checkpoint epoch",
    CHECKPOINT_EPOCH,
    "berhasil dimuat."
)

print(
    "✅ Forward pass berhasil."
)

print(
    "✅ CRF Viterbi decode berhasil."
)

print(
    "✅ Panjang output sama dengan input."
)

print(
    "✅ Cell 5 selesai."
)

print("=" * 70)


In [ ]:
# ============================================================
# CELL 6 — PREPROCESSING & VALIDASI INPUT
# ============================================================

import re
import unicodedata


print("=" * 70)
print("PREPROCESSING & VALIDIDASI INPUT")
print("=" * 70)


# ------------------------------------------------------------
# 1. Deteksi harakat Arab
#
# Hanya Unicode combining mark kategori Mn
# dengan nama ARABIC yang dianggap diakritik.
#
# Ini penting agar tanda seperti:
# U+06DD ARABIC END OF AYAH
# TIDAK salah dianggap sebagai harakat.
# ------------------------------------------------------------

def is_arabic_diacritic(char):

    if not char:
        return False

    if unicodedata.category(char) != "Mn":
        return False

    name = unicodedata.name(
        char,
        ""
    )

    return (
        "ARABIC"
        in name
    )


# ------------------------------------------------------------
# 2. Hapus harakat Arab
# ------------------------------------------------------------

def remove_arabic_diacritics(text):

    if text is None:
        return ""

    return "".join(
        char
        for char in str(text)
        if not is_arabic_diacritic(char)
    )


# ------------------------------------------------------------
# 3. Cek apakah karakter huruf Arab
# ------------------------------------------------------------

def is_arabic_letter(char):

    if not char:
        return False

    category = unicodedata.category(
        char
    )

    name = unicodedata.name(
        char,
        ""
    )

    return (
        category.startswith("L")
        and
        "ARABIC"
        in name
    )


# ------------------------------------------------------------
# 4. Cek huruf Latin
# ------------------------------------------------------------

def is_latin_letter(char):

    if not char:
        return False

    category = unicodedata.category(
        char
    )

    name = unicodedata.name(
        char,
        ""
    )

    return (
        category.startswith("L")
        and
        "LATIN"
        in name
    )


# ------------------------------------------------------------
# 5. Normalisasi Unicode
#
# Tidak melakukan stemming, transliterasi,
# atau perubahan huruf.
# ------------------------------------------------------------

def normalize_text(text):

    if text is None:
        return ""

    text = unicodedata.normalize(
        "NFC",
        str(text),
    )

    # Samakan normalisasi inferensi dengan prapemrosesan data pelatihan.
    text = text.replace("ـ", "")
    text = text.replace("\u0670", "")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def count_arabic_letters(text):

    return sum(
        1
        for char in text
        if is_arabic_letter(char)
    )


# ------------------------------------------------------------
# 7. Cari huruf Latin
# ------------------------------------------------------------

def find_latin_letters(text):

    return sorted(
        set(
            char
            for char in text
            if is_latin_letter(char)
        )
    )


# ------------------------------------------------------------
# 8. Cek apakah teks memiliki harakat
# ------------------------------------------------------------

def contains_arabic_diacritics(text):

    return any(
        is_arabic_diacritic(char)
        for char in text
    )


# ------------------------------------------------------------
# 9. Validasi input manual
# ------------------------------------------------------------

def validate_input(text):

    # --------------------------------------------
    # Normalisasi
    # --------------------------------------------

    normalized = normalize_text(
        text
    )


    # --------------------------------------------
    # Input kosong
    # --------------------------------------------

    if not normalized.strip():

        return {
            "valid": False,
            "message":
                "Input masih kosong.",
            "normalized_text": normalized,
            "clean_text": "",
            "had_diacritics": False,
            "arabic_letters": 0,
            "latin_letters": [],
        }


    # --------------------------------------------
    # Deteksi Latin
    # --------------------------------------------

    latin_letters = find_latin_letters(
        normalized
    )


    if latin_letters:

        return {
            "valid": False,
            "message": (
                "Input manual mengandung huruf Latin. "
                "Masukkan teks Arab tanpa huruf Latin."
            ),
            "normalized_text": normalized,
            "clean_text": "",
            "had_diacritics":
                contains_arabic_diacritics(
                    normalized
                ),
            "arabic_letters":
                count_arabic_letters(
                    normalized
                ),
            "latin_letters":
                latin_letters,
        }


    # --------------------------------------------
    # Cek apakah input mempunyai huruf Arab
    # --------------------------------------------

    arabic_letters = count_arabic_letters(
        normalized
    )


    if arabic_letters == 0:

        return {
            "valid": False,
            "message":
                "Tidak ditemukan huruf Arab pada input.",
            "normalized_text": normalized,
            "clean_text": "",
            "had_diacritics": False,
            "arabic_letters": 0,
            "latin_letters": [],
        }


    # --------------------------------------------
    # Deteksi harakat lama
    # --------------------------------------------

    had_diacritics = (
        contains_arabic_diacritics(
            normalized
        )
    )


    # --------------------------------------------
    # Hapus harakat lama sebelum inference
    # --------------------------------------------

    clean_text = (
        remove_arabic_diacritics(
            normalized
        )
    )


    # Normalisasi lagi setelah penghapusan mark
    clean_text = unicodedata.normalize(
        "NFC",
        clean_text
    )


    # --------------------------------------------
    # Hasil
    # --------------------------------------------

    return {
        "valid": True,
        "message":
            "Input valid.",
        "normalized_text":
            normalized,
        "clean_text":
            clean_text,
        "had_diacritics":
            had_diacritics,
        "arabic_letters":
            arabic_letters,
        "latin_letters":
            [],
    }


# ============================================================
# 10. TEST
# ============================================================

print("\nTEST FUNGSI")
print("-" * 70)


tests = {

    "Arab gundul":
        "قال الشيخ الإمام العالم",

    "Sudah berharakat":
        "قَالَ الشَّيْخُ الْإِمَامُ",

    "Latin":
        "hello world",

    "Campuran":
        "قال الشيخ hello",

    "Kosong":
        "   ",

    "Angka saja":
        "12345",

    "Arab + angka":
        "قال الشيخ 123",
}


for name, text in tests.items():

    result = validate_input(
        text
    )

    print(
        f"\n{name}"
    )

    print(
        "Input          :",
        repr(text)
    )

    print(
        "Valid          :",
        result["valid"]
    )

    print(
        "Had diacritics :",
        result["had_diacritics"]
    )

    print(
        "Huruf Arab     :",
        result["arabic_letters"]
    )

    print(
        "Clean text     :",
        repr(
            result["clean_text"]
        )
    )

    print(
        "Message        :",
        result["message"]
    )


# ============================================================
# 11. TEST KHUSUS UNICODE
# ============================================================

print("\nTEST UNICODE")
print("-" * 70)


# Fathah harus dianggap harakat
assert is_arabic_diacritic(
    "\u064E"
) is True


# Kasrah
assert is_arabic_diacritic(
    "\u0650"
) is True


# Dammah
assert is_arabic_diacritic(
    "\u064F"
) is True


# Shadda
assert is_arabic_diacritic(
    "\u0651"
) is True


# ARABIC END OF AYAH
# BUKAN combining mark / harakat
assert is_arabic_diacritic(
    "\u06DD"
) is False


print(
    "✅ Harakat Unicode terdeteksi dengan benar."
)

print(
    "✅ U+06DD ARABIC END OF AYAH tidak dianggap harakat."
)


# ============================================================
# 12. TEST PENGHAPUSAN HARAKAT
# ============================================================

reference = (
    "قَالَ الشَّيْخُ الْإِمَامُ"
)

expected = (
    "قال الشيخ الإمام"
)

cleaned = remove_arabic_diacritics(
    reference
)


print("\nTEST REMOVE HARAKAT")
print("-" * 70)

print(
    "Berharakat :",
    reference
)

print(
    "Gundul     :",
    cleaned
)


assert cleaned == expected, (
    "❌ Penghapusan harakat tidak sesuai."
)


print(
    "✅ Penghapusan harakat sesuai."
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)
print("✅ Preprocessing berhasil.")
print("✅ Validasi input berhasil.")
print("✅ Teks berharakat dapat dibersihkan.")
print("✅ Huruf Latin dapat ditolak.")
print("✅ Unicode harakat ditangani dengan aman.")
print("✅ Cell 6 selesai.")
print("=" * 70)

In [ ]:
# ============================================================
# CELL 7 — AUTO-CHUNKING TEKS PANJANG
# ============================================================

print("=" * 70)
print("AUTO-CHUNKING TEKS")
print("=" * 70)


# ------------------------------------------------------------
# 1. Fungsi chunking
#
# Prioritas:
# - maksimal 300 karakter
# - potong pada whitespace terakhir jika memungkinkan
# - tidak membuang spasi/newline
# - hasil chunk jika digabung harus sama persis dengan input
# ------------------------------------------------------------

def split_long_text(
    text,
    max_len=MAX_SEQUENCE_LENGTH,
):

    text = normalize_text(text)

    if not text:
        return []

    if max_len <= 0:

        raise ValueError(
            "max_len harus lebih besar dari 0."
        )


    # Jika sudah pendek
    if len(text) <= max_len:

        return [text]


    chunks = []

    start = 0
    text_length = len(text)


    while start < text_length:

        end = min(
            start + max_len,
            text_length,
        )


        # --------------------------------------------
        # Sisa teks sudah <= max_len
        # --------------------------------------------

        if end == text_length:

            chunk = text[
                start:end
            ]

            if chunk:
                chunks.append(
                    chunk
                )

            break


        # --------------------------------------------
        # Cari whitespace terakhir dalam window
        # --------------------------------------------

        split_position = None


        for index in range(
            end - 1,
            start,
            -1,
        ):

            if text[index].isspace():

                # whitespace ikut chunk sebelumnya
                split_position = (
                    index + 1
                )

                break


        # --------------------------------------------
        # Kalau tidak ada whitespace,
        # lakukan hard split max_len.
        #
        # Kasus ini jarang dan biasanya hanya terjadi
        # jika ada token/kata sangat panjang.
        # --------------------------------------------

        if split_position is None:

            split_position = end


        chunk = text[
            start:split_position
        ]


        if chunk:

            chunks.append(
                chunk
            )


        start = split_position


    return chunks


# ============================================================
# 2. TEST TEKS PENDEK
# ============================================================

short_text = (
    "قال الشيخ الإمام العالم العلامة شمس الدين"
)

short_chunks = split_long_text(
    short_text
)


print("\nTEST TEKS PENDEK")
print("-" * 70)

print(
    "Panjang:",
    len(short_text)
)

print(
    "Jumlah chunk:",
    len(short_chunks)
)

print(
    "Chunk:",
    short_chunks
)


assert len(short_chunks) == 1

assert short_chunks[0] == short_text


print(
    "✅ Teks pendek tidak dipotong."
)


# ============================================================
# 3. TEST TEKS PANJANG
# ============================================================

base_text = (
    "قال الشيخ الإمام العالم العلامة شمس الدين "
    "رحمه الله تعالى ونفعنا بعلومه في الدارين "
)

long_text = (
    base_text * 20
).strip()


long_chunks = split_long_text(
    long_text
)


print("\nTEST TEKS PANJANG")
print("-" * 70)

print(
    "Panjang total:",
    len(long_text)
)

print(
    "Jumlah chunk:",
    len(long_chunks)
)

print(
    "Panjang tiap chunk:",
    [
        len(chunk)
        for chunk in long_chunks
    ]
)


# ------------------------------------------------------------
# Semua <= max sequence
# ------------------------------------------------------------

assert all(
    len(chunk)
    <= MAX_SEQUENCE_LENGTH

    for chunk in long_chunks
), (
    "❌ Ada chunk melebihi batas 300 karakter."
)


# ------------------------------------------------------------
# Tidak boleh ada chunk kosong
# ------------------------------------------------------------

assert all(
    len(chunk) > 0

    for chunk in long_chunks
), (
    "❌ Ada chunk kosong."
)


# ------------------------------------------------------------
# Rekonstruksi HARUS sama persis
# ------------------------------------------------------------

reconstructed = "".join(
    long_chunks
)


assert reconstructed == long_text, (
    "❌ Hasil penggabungan chunk berbeda "
    "dengan teks asli."
)


print(
    "✅ Semua chunk <= ",
    MAX_SEQUENCE_LENGTH,
    " karakter.",
    sep="",
)

print(
    "✅ Tidak ada chunk kosong."
)

print(
    "✅ Rekonstruksi sama persis dengan input."
)


# ============================================================
# 4. TEST NEWLINE
# ============================================================

multiline_text = (
    "قال الشيخ الإمام العالم\n"
    "العلامة شمس الدين\n\n"
    "الحمد لله رب العالمين"
)


multiline_chunks = split_long_text(
    multiline_text,
    max_len=25,
)


print("\nTEST NEWLINE")
print("-" * 70)

print(
    "Jumlah chunk:",
    len(multiline_chunks)
)

print(
    "Panjang:",
    [
        len(chunk)
        for chunk in multiline_chunks
    ]
)


assert "".join(
    multiline_chunks
) == multiline_text


print(
    "✅ Newline dan blank line tetap dipertahankan."
)


# ============================================================
# 5. TEST TANPA WHITESPACE
# ============================================================

no_space_text = (
    "ا" * 650
)


no_space_chunks = split_long_text(
    no_space_text
)


print("\nTEST TANPA WHITESPACE")
print("-" * 70)

print(
    "Panjang:",
    [
        len(chunk)
        for chunk in no_space_chunks
    ]
)


assert [
    len(chunk)
    for chunk in no_space_chunks
] == [
    300,
    300,
    50,
]


assert "".join(
    no_space_chunks
) == no_space_text


print(
    "✅ Hard split bekerja jika tidak ada whitespace."
)


# ============================================================
# 6. TEST DATA REALISTIS
# ============================================================

realistic_text = (
    "قال الشيخ الإمام العالم العلامة شمس الدين "
    * 30
).strip()


realistic_chunks = split_long_text(
    realistic_text
)


print("\nTEST REALISTIS")
print("-" * 70)

for index, chunk in enumerate(
    realistic_chunks,
    start=1,
):

    print(
        f"Chunk {index:>2} : "
        f"{len(chunk):>3} karakter | "
        f"{repr(chunk[:30])}..."
    )


assert max(
    len(chunk)
    for chunk in realistic_chunks
) <= MAX_SEQUENCE_LENGTH


assert "".join(
    realistic_chunks
) == realistic_text


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)

print(
    "✅ Auto-chunking berhasil."
)

print(
    f"✅ Batas maksimum: "
    f"{MAX_SEQUENCE_LENGTH} karakter."
)

print(
    "✅ Pemotongan memprioritaskan whitespace."
)

print(
    "✅ Spasi dan newline tidak hilang."
)

print(
    "✅ Rekonstruksi teks tetap identik."
)

print(
    "✅ Cell 7 selesai."
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 8 — INFERENCE SATU CHUNK
# ============================================================

import unicodedata
import torch


print("=" * 70)
print("INFERENCE SATU CHUNK")
print("=" * 70)


# ------------------------------------------------------------
# 1. Konversi label menjadi tanda harakat
# ------------------------------------------------------------

def label_to_diacritics(label):

    if label == "O":
        return ""

    # Label kombinasi contoh:
    # َ|ّ
    # ُ|ّ
    # ِ|ّ
    return "".join(
        part
        for part in label.split("|")
        if part
    )


# ------------------------------------------------------------
# 2. Rekonstruksi karakter + label
# ------------------------------------------------------------

def reconstruct_diacritized_text(
    clean_text,
    predicted_label_ids,
):

    if len(clean_text) != len(
        predicted_label_ids
    ):

        raise ValueError(
            "Panjang teks dan label prediksi berbeda."
        )


    output = []


    for char, label_id in zip(
        clean_text,
        predicted_label_ids,
    ):

        label_id = int(
            label_id
        )


        if label_id not in id2label:

            raise KeyError(
                f"ID label tidak dikenal: {label_id}"
            )


        label = id2label[
            label_id
        ]


        # --------------------------------------------
        # Karakter asli selalu dipertahankan
        # --------------------------------------------

        output.append(
            char
        )


        # --------------------------------------------
        # Harakat hanya ditempelkan pada huruf Arab.
        #
        # Spasi, angka, punctuation, newline, dll
        # tetap dipertahankan tetapi tidak diberi
        # harakat meskipun model secara tidak sengaja
        # memprediksi label non-O.
        # --------------------------------------------

        if is_arabic_letter(
            char
        ):

            diacritics = (
                label_to_diacritics(
                    label
                )
            )

            if diacritics:

                output.append(
                    diacritics
                )


    result = "".join(
        output
    )


    # Normalisasi Unicode hasil akhir
    return unicodedata.normalize(
        "NFC",
        result
    )


# ------------------------------------------------------------
# 3. Fungsi inference satu chunk
# ------------------------------------------------------------

def predict_single_chunk(
    text,
):

    # --------------------------------------------
    # Normalisasi
    # --------------------------------------------

    normalized = normalize_text(
        text
    )


    if not normalized:

        return {
            "success": False,
            "message": "Chunk kosong.",
            "input_text": "",
            "clean_text": "",
            "prediction": "",
            "predicted_label_ids": [],
            "predicted_labels": [],
            "unknown_characters": [],
            "length": 0,
        }


    # --------------------------------------------
    # Hapus harakat lama
    # --------------------------------------------

    clean_text = (
        remove_arabic_diacritics(
            normalized
        )
    )

    clean_text = unicodedata.normalize(
        "NFC",
        clean_text
    )


    if not clean_text:

        return {
            "success": False,
            "message":
                "Tidak ada karakter yang dapat diproses.",
            "input_text": normalized,
            "clean_text": "",
            "prediction": "",
            "predicted_label_ids": [],
            "predicted_labels": [],
            "unknown_characters": [],
            "length": 0,
        }


    # --------------------------------------------
    # Maksimal 300 karakter
    # --------------------------------------------

    if len(clean_text) > MAX_SEQUENCE_LENGTH:

        raise ValueError(
            f"Chunk terlalu panjang: "
            f"{len(clean_text)} karakter. "
            f"Maksimum {MAX_SEQUENCE_LENGTH}."
        )


    # --------------------------------------------
    # Encode karakter
    # --------------------------------------------

    input_ids = []

    unknown_characters = []


    for char in clean_text:

        if char in char2id:

            char_id = char2id[
                char
            ]

        else:

            char_id = UNK_ID

            if char not in unknown_characters:

                unknown_characters.append(
                    char
                )


        input_ids.append(
            char_id
        )


    # --------------------------------------------
    # Tensor
    # --------------------------------------------

    input_tensor = torch.tensor(
        [input_ids],
        dtype=torch.long,
        device=DEVICE,
    )


    lengths = torch.tensor(
        [len(input_ids)],
        dtype=torch.long,
    )


    # Tidak ada padding pada single chunk.
    # Semua posisi valid.
    mask = torch.ones(
        (
            1,
            len(input_ids),
        ),
        dtype=torch.bool,
        device=DEVICE,
    )


    # --------------------------------------------
    # Inference
    # --------------------------------------------

    model.eval()


    with torch.inference_mode():

        decoded = model.decode(
            input_tensor,
            mask,
            lengths,
        )


    if len(decoded) != 1:

        raise RuntimeError(
            "CRF menghasilkan jumlah sequence "
            "yang tidak sesuai."
        )


    predicted_label_ids = decoded[
        0
    ]


    # --------------------------------------------
    # Validasi panjang
    # --------------------------------------------

    if len(
        predicted_label_ids
    ) != len(
        clean_text
    ):

        raise RuntimeError(
            "Jumlah label hasil decode berbeda "
            "dengan jumlah karakter input."
        )


    # --------------------------------------------
    # ID → label
    # --------------------------------------------

    predicted_labels = [

        id2label[
            int(label_id)
        ]

        for label_id
        in predicted_label_ids
    ]


    # --------------------------------------------
    # Rekonstruksi teks
    # --------------------------------------------

    prediction = (
        reconstruct_diacritized_text(
            clean_text,
            predicted_label_ids,
        )
    )


    return {
        "success": True,

        "message":
            "Inference berhasil.",

        "input_text":
            normalized,

        "clean_text":
            clean_text,

        "prediction":
            prediction,

        "predicted_label_ids":
            predicted_label_ids,

        "predicted_labels":
            predicted_labels,

        "unknown_characters":
            unknown_characters,

        "length":
            len(clean_text),
    }


# ============================================================
# 4. TEST MODEL
# ============================================================

TEST_TEXT = (
    "قال الشيخ الإمام العالم العلامة شمس الدين"
)


result = predict_single_chunk(
    TEST_TEXT
)


print("\nHASIL TEST")
print("-" * 70)

print(
    "Success :",
    result["success"]
)

print(
    "Input   :",
    result["input_text"]
)

print(
    "Clean   :",
    result["clean_text"]
)

print(
    "Output  :",
    result["prediction"]
)

print(
    "Panjang :",
    result["length"]
)

print(
    "Label   :",
    len(
        result[
            "predicted_labels"
        ]
    )
)

print(
    "Unknown :",
    result[
        "unknown_characters"
    ]
)


# ------------------------------------------------------------
# 5. Validasi
# ------------------------------------------------------------

assert result[
    "success"
] is True


assert result[
    "clean_text"
] == TEST_TEXT


assert result[
    "length"
] == len(
    TEST_TEXT
)


assert len(
    result[
        "predicted_labels"
    ]
) == len(
    TEST_TEXT
)


assert len(
    result[
        "predicted_label_ids"
    ]
) == len(
    TEST_TEXT
)


# ------------------------------------------------------------
# 6. Test input sudah berharakat
# ------------------------------------------------------------

TEST_DIACRITIZED = (
    "قَالَ الشَّيْخُ الْإِمَامُ"
)


result_diacritized = (
    predict_single_chunk(
        TEST_DIACRITIZED
    )
)


print("\nTEST INPUT SUDAH BERHARAKAT")
print("-" * 70)

print(
    "Input  :",
    TEST_DIACRITIZED
)

print(
    "Clean  :",
    result_diacritized[
        "clean_text"
    ]
)

print(
    "Output :",
    result_diacritized[
        "prediction"
    ]
)


assert (
    result_diacritized[
        "clean_text"
    ]
    ==
    "قال الشيخ الإمام"
)


# ------------------------------------------------------------
# 7. Test panjang > 300 harus ditolak
# ------------------------------------------------------------

print("\nTEST BATAS PANJANG")
print("-" * 70)

try:

    predict_single_chunk(
        "ا" * 301
    )

    raise AssertionError(
        "❌ Teks >300 seharusnya ditolak."
    )

except ValueError:

    print(
        "✅ Chunk >300 karakter ditolak."
    )


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)

print(
    "✅ Encoding karakter berhasil."
)

print(
    "✅ BiLSTM inference berhasil."
)

print(
    "✅ CRF Viterbi decoding berhasil."
)

print(
    "✅ Rekonstruksi harakat berhasil."
)

print(
    "✅ Input berharakat dapat diproses ulang."
)

print(
    "✅ Batas 300 karakter diterapkan."
)

print(
    "✅ Cell 8 selesai."
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 9 — INFERENCE TEKS PANJANG
# ============================================================

print("=" * 70)
print("INFERENCE TEKS PANJANG")
print("=" * 70)


# ------------------------------------------------------------
# 1. Fungsi inference teks panjang
# ------------------------------------------------------------

def predict_long_text(
    text,
    progress_callback=None,
):

    # --------------------------------------------------------
    # Validasi input
    # --------------------------------------------------------

    validation = validate_input(
        text
    )


    if not validation["valid"]:

        return {
            "success": False,
            "message":
                validation["message"],

            "input_text":
                validation["normalized_text"],

            "clean_text":
                validation["clean_text"],

            "prediction":
                "",

            "had_diacritics":
                validation["had_diacritics"],

            "arabic_letters":
                validation["arabic_letters"],

            "total_characters":
                0,

            "total_chunks":
                0,

            "chunk_lengths":
                [],

            "unknown_characters":
                [],
        }


    # --------------------------------------------------------
    # Teks bersih tanpa harakat
    # --------------------------------------------------------

    clean_text = validation[
        "clean_text"
    ]


    # --------------------------------------------------------
    # Auto-chunking
    # --------------------------------------------------------

    chunks = split_long_text(
        clean_text,
        max_len=MAX_SEQUENCE_LENGTH,
    )


    if not chunks:

        return {
            "success": False,
            "message":
                "Tidak ada teks yang dapat diproses.",

            "input_text":
                validation["normalized_text"],

            "clean_text":
                clean_text,

            "prediction":
                "",

            "had_diacritics":
                validation["had_diacritics"],

            "arabic_letters":
                validation["arabic_letters"],

            "total_characters":
                len(clean_text),

            "total_chunks":
                0,

            "chunk_lengths":
                [],

            "unknown_characters":
                [],
        }


    # --------------------------------------------------------
    # Inference per chunk
    # --------------------------------------------------------

    predictions = []

    unknown_characters = []

    total_chunks = len(
        chunks
    )


    for index, chunk in enumerate(
        chunks,
        start=1,
    ):

        chunk_result = (
            predict_single_chunk(
                chunk
            )
        )


        if not chunk_result[
            "success"
        ]:

            return {
                "success": False,

                "message": (
                    f"Gagal memproses chunk "
                    f"{index}/{total_chunks}: "
                    f"{chunk_result['message']}"
                ),

                "input_text":
                    validation[
                        "normalized_text"
                    ],

                "clean_text":
                    clean_text,

                "prediction":
                    "",

                "had_diacritics":
                    validation[
                        "had_diacritics"
                    ],

                "arabic_letters":
                    validation[
                        "arabic_letters"
                    ],

                "total_characters":
                    len(clean_text),

                "total_chunks":
                    total_chunks,

                "chunk_lengths":
                    [
                        len(item)
                        for item in chunks
                    ],

                "unknown_characters":
                    unknown_characters,
            }


        predictions.append(
            chunk_result[
                "prediction"
            ]
        )


        # ----------------------------------------------------
        # Gabungkan unknown character tanpa duplikat
        # ----------------------------------------------------

        for char in chunk_result[
            "unknown_characters"
        ]:

            if char not in unknown_characters:

                unknown_characters.append(
                    char
                )


        # ----------------------------------------------------
        # Callback untuk UI progress bar
        # ----------------------------------------------------

        if progress_callback is not None:

            progress_callback(
                index,
                total_chunks,
            )


    # --------------------------------------------------------
    # Gabungkan TANPA menambah spasi baru
    #
    # Karena split_long_text mempertahankan whitespace asli.
    # --------------------------------------------------------

    prediction = "".join(
        predictions
    )


    prediction = unicodedata.normalize(
        "NFC",
        prediction
    )


    # --------------------------------------------------------
    # Validasi struktur hasil
    #
    # Jika seluruh harakat hasil prediction dibuang,
    # harus kembali persis ke clean_text.
    # --------------------------------------------------------

    reconstructed_clean = (
        remove_arabic_diacritics(
            prediction
        )
    )


    if reconstructed_clean != clean_text:

        raise RuntimeError(
            "❌ Rekonstruksi hasil inference "
            "tidak sama dengan teks input bersih."
        )


    # --------------------------------------------------------
    # Return
    # --------------------------------------------------------

    return {
        "success": True,

        "message":
            "Diakritisasi berhasil.",

        "input_text":
            validation[
                "normalized_text"
            ],

        "clean_text":
            clean_text,

        "prediction":
            prediction,

        "had_diacritics":
            validation[
                "had_diacritics"
            ],

        "arabic_letters":
            validation[
                "arabic_letters"
            ],

        "total_characters":
            len(clean_text),

        "total_chunks":
            total_chunks,

        "chunk_lengths":
            [
                len(chunk)
                for chunk in chunks
            ],

        "unknown_characters":
            unknown_characters,
    }


# ============================================================
# 2. TEST TEKS PENDEK
# ============================================================

short_text = (
    "قال الشيخ الإمام العالم العلامة شمس الدين"
)


short_result = predict_long_text(
    short_text
)


print("\nTEST TEKS PENDEK")
print("-" * 70)

print(
    "Success :",
    short_result["success"]
)

print(
    "Input   :",
    short_text
)

print(
    "Output  :",
    short_result["prediction"]
)

print(
    "Chunk   :",
    short_result["total_chunks"]
)

print(
    "Length  :",
    short_result["chunk_lengths"]
)

print(
    "Unknown :",
    short_result["unknown_characters"]
)


assert short_result[
    "success"
] is True

assert short_result[
    "total_chunks"
] == 1

assert (
    remove_arabic_diacritics(
        short_result[
            "prediction"
        ]
    )
    ==
    short_result[
        "clean_text"
    ]
)


print(
    "✅ Inference teks pendek berhasil."
)


# ============================================================
# 3. TEST TEKS PANJANG
# ============================================================

base_text = (
    "قال الشيخ الإمام العالم العلامة شمس الدين "
    "رحمه الله تعالى ونفعنا بعلومه في الدارين. "
)


long_text = (
    base_text * 15
).strip()


# ------------------------------------------------------------
# Progress callback test
# ------------------------------------------------------------

progress_events = []


def test_progress_callback(
    current,
    total,
):

    progress_events.append(
        (
            current,
            total,
        )
    )


long_result = predict_long_text(
    long_text,
    progress_callback=
        test_progress_callback,
)


print("\nTEST TEKS PANJANG")
print("-" * 70)

print(
    "Success          :",
    long_result[
        "success"
    ]
)

print(
    "Karakter         :",
    long_result[
        "total_characters"
    ]
)

print(
    "Huruf Arab       :",
    long_result[
        "arabic_letters"
    ]
)

print(
    "Jumlah chunk     :",
    long_result[
        "total_chunks"
    ]
)

print(
    "Panjang chunk    :",
    long_result[
        "chunk_lengths"
    ]
)

print(
    "Unknown          :",
    long_result[
        "unknown_characters"
    ]
)

print(
    "Progress events  :",
    progress_events
)


# ------------------------------------------------------------
# Validasi panjang chunk
# ------------------------------------------------------------

assert long_result[
    "success"
] is True


assert long_result[
    "total_chunks"
] > 1


assert all(
    length
    <= MAX_SEQUENCE_LENGTH

    for length
    in long_result[
        "chunk_lengths"
    ]
)


# ------------------------------------------------------------
# Tidak boleh kehilangan karakter dasar
# ------------------------------------------------------------

assert (
    remove_arabic_diacritics(
        long_result[
            "prediction"
        ]
    )
    ==
    long_result[
        "clean_text"
    ]
)


# ------------------------------------------------------------
# Callback harus dipanggil sebanyak jumlah chunk
# ------------------------------------------------------------

assert len(
    progress_events
) == long_result[
    "total_chunks"
]


assert progress_events[-1] == (
    long_result[
        "total_chunks"
    ],
    long_result[
        "total_chunks"
    ],
)


print(
    "✅ Inference teks panjang berhasil."
)

print(
    "✅ Tidak ada karakter dasar yang hilang."
)

print(
    "✅ Progress callback berhasil."
)


# ============================================================
# 4. TEST INPUT SUDAH BERHARAKAT
# ============================================================

diacritized_input = (
    "قَالَ الشَّيْخُ الْإِمَامُ "
    "الْعَالِمُ الْعَلَّامَةُ"
)


diacritized_result = (
    predict_long_text(
        diacritized_input
    )
)


print("\nTEST INPUT BERHARAKAT")
print("-" * 70)

print(
    "Had diacritics :",
    diacritized_result[
        "had_diacritics"
    ]
)

print(
    "Clean          :",
    diacritized_result[
        "clean_text"
    ]
)

print(
    "Output         :",
    diacritized_result[
        "prediction"
    ]
)


assert diacritized_result[
    "success"
] is True


assert diacritized_result[
    "had_diacritics"
] is True


assert not contains_arabic_diacritics(
    diacritized_result[
        "clean_text"
    ]
)


print(
    "✅ Input berharakat dibersihkan "
    "dan diprediksi ulang."
)


# ============================================================
# 5. TEST INPUT INVALID
# ============================================================

invalid_result = predict_long_text(
    "hello world"
)


print("\nTEST INPUT INVALID")
print("-" * 70)

print(
    "Success :",
    invalid_result[
        "success"
    ]
)

print(
    "Message :",
    invalid_result[
        "message"
    ]
)


assert invalid_result[
    "success"
] is False


print(
    "✅ Input Latin ditolak."
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)

print(
    "✅ predict_long_text berhasil."
)

print(
    "✅ Auto-chunking terintegrasi."
)

print(
    "✅ Inference per chunk berhasil."
)

print(
    "✅ Output digabung tanpa kehilangan whitespace."
)

print(
    "✅ Teks panjang dapat diproses."
)

print(
    "✅ Progress callback siap untuk Streamlit."
)

print(
    "✅ Cell 9 selesai."
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 10 — PIPELINE FILE TXT + CSV
# ============================================================

from io import BytesIO, StringIO
from pathlib import Path
from collections import Counter

import pandas as pd


print("=" * 70)
print("PIPELINE FILE TXT + CSV")
print("=" * 70)


MAX_FILE_SIZE = 5 * 1024 * 1024   # 5 MB


# ============================================================
# 1. AMBIL BYTES FILE
# ============================================================

def get_uploaded_file_bytes(uploaded_file):

    if uploaded_file is None:
        return b""

    if isinstance(uploaded_file, bytes):
        return uploaded_file

    if isinstance(uploaded_file, bytearray):
        return bytes(uploaded_file)

    if hasattr(uploaded_file, "getvalue"):
        return uploaded_file.getvalue()

    if hasattr(uploaded_file, "read"):

        position = None

        if hasattr(uploaded_file, "tell"):
            try:
                position = uploaded_file.tell()
            except Exception:
                pass

        data = uploaded_file.read()

        if position is not None and hasattr(
            uploaded_file,
            "seek"
        ):
            try:
                uploaded_file.seek(position)
            except Exception:
                pass

        return data

    raise TypeError(
        "Format file tidak didukung."
    )


# ============================================================
# 2. DECODE FILE TEKS
# ============================================================

def decode_text_bytes(data):

    encodings = [
        "utf-8-sig",
        "utf-8",
        "utf-16",
        "cp1256",
    ]

    for encoding in encodings:

        try:

            return {
                "success": True,
                "text": data.decode(
                    encoding
                ),
                "encoding": encoding,
                "message": (
                    f"File berhasil dibaca "
                    f"dengan encoding {encoding}."
                ),
            }

        except UnicodeDecodeError:
            continue

    return {
        "success": False,
        "text": "",
        "encoding": None,
        "message": (
            "Encoding file tidak dikenali. "
            "Gunakan UTF-8 jika memungkinkan."
        ),
    }


# ============================================================
# 3. KLASIFIKASI UNIT TEKS
# ============================================================

def classify_text_unit(text):

    text = normalize_text(
        text
    )

    if not text.strip():
        return "kosong"

    has_arabic = any(
        is_arabic_letter(char)
        for char in text
    )

    has_latin = any(
        is_latin_letter(char)
        for char in text
    )

    if has_arabic and has_latin:
        return "campuran"

    if has_arabic:
        return "arab"

    return "non_arab"


# ============================================================
# 4. PROSES SATU UNIT TEKS
# ============================================================

def process_text_unit(text):

    normalized = normalize_text(
        text
    )

    category = classify_text_unit(
        normalized
    )


    # --------------------------------------------------------
    # Kosong
    # --------------------------------------------------------

    if category == "kosong":

        return {
            "success": True,
            "status": "kosong",
            "input": normalized,
            "prediction": normalized,
            "note": "Teks kosong dipertahankan.",
            "unknown_characters": [],
        }


    # --------------------------------------------------------
    # Non Arab
    # --------------------------------------------------------

    if category == "non_arab":

        return {
            "success": True,
            "status": "non_arab",
            "input": normalized,
            "prediction": normalized,
            "note": (
                "Tidak mengandung huruf Arab; "
                "teks dipertahankan."
            ),
            "unknown_characters": [],
        }


    # --------------------------------------------------------
    # Campuran Arab + Latin
    #
    # Tidak diproses agar isi asli tidak rusak.
    # --------------------------------------------------------

    if category == "campuran":

        return {
            "success": True,
            "status": "campuran",
            "input": normalized,
            "prediction": normalized,
            "note": (
                "Teks mengandung campuran huruf Arab "
                "dan Latin; dipertahankan tanpa prediksi."
            ),
            "unknown_characters": [],
        }


    # --------------------------------------------------------
    # Arab
    # --------------------------------------------------------

    result = predict_long_text(
        normalized
    )


    if not result[
        "success"
    ]:

        return {
            "success": False,
            "status": "gagal",
            "input": normalized,
            "prediction": normalized,
            "note": result[
                "message"
            ],
            "unknown_characters": [],
        }


    note = (
        "Diakritisasi berhasil."
    )

    if result[
        "had_diacritics"
    ]:

        note = (
            "Harakat lama dihapus dan teks "
            "diprediksi ulang."
        )


    return {
        "success": True,
        "status": "berhasil",
        "input": normalized,
        "prediction": result[
            "prediction"
        ],
        "note": note,
        "unknown_characters": result[
            "unknown_characters"
        ],
    }


# ============================================================
# 5. PISAHKAN LINE ENDING
# ============================================================

def split_line_ending(line):

    if line.endswith("\r\n"):
        return line[:-2], "\r\n"

    if line.endswith("\n"):
        return line[:-1], "\n"

    if line.endswith("\r"):
        return line[:-1], "\r"

    return line, ""


# ============================================================
# 6. PROSES TXT
# ============================================================

def process_txt_bytes(
    data,
    filename="input.txt",
    progress_callback=None,
):

    decoded = decode_text_bytes(
        data
    )


    if not decoded[
        "success"
    ]:

        return {
            "success": False,
            "message": decoded[
                "message"
            ],
        }


    text = decoded[
        "text"
    ]


    lines = text.splitlines(
        keepends=True
    )


    # File tanpa newline tetap dianggap 1 baris
    if not lines and text:

        lines = [
            text
        ]


    output_parts = []
    statuses = []
    warnings = []
    unknown_characters = []


    total_lines = len(
        lines
    )


    for index, raw_line in enumerate(
        lines,
        start=1,
    ):

        content, line_ending = (
            split_line_ending(
                raw_line
            )
        )


        result = process_text_unit(
            content
        )


        output_parts.append(
            result[
                "prediction"
            ]
            +
            line_ending
        )


        statuses.append(
            result[
                "status"
            ]
        )


        for char in result.get(
            "unknown_characters",
            [],
        ):

            if char not in unknown_characters:

                unknown_characters.append(
                    char
                )


        if result[
            "status"
        ] == "campuran":

            warnings.append({
                "line": index,
                "note": result[
                    "note"
                ],
            })


        if progress_callback is not None:

            progress_callback(
                index,
                total_lines,
            )


    prediction = "".join(
        output_parts
    )


    status_summary = dict(
        Counter(
            statuses
        )
    )


    output_filename = (
        f"{Path(filename).stem}"
        "_hasil_diakritisasi.txt"
    )


    return {
        "success": True,
        "message": (
            "File TXT berhasil diproses."
        ),
        "input_type": "txt",
        "encoding": decoded[
            "encoding"
        ],
        "prediction": prediction,
        "status_summary": status_summary,
        "warnings": warnings,
        "unknown_characters": unknown_characters,
        "output_filename": output_filename,
        "download_bytes": prediction.encode(
            "utf-8"
        ),
    }


# ============================================================
# 7. BACA CSV
# ============================================================

def read_csv_bytes(data):

    decoded = decode_text_bytes(
        data
    )


    if not decoded[
        "success"
    ]:

        return {
            "success": False,
            "message": decoded[
                "message"
            ],
        }


    try:

        dataframe = pd.read_csv(
            StringIO(
                decoded[
                    "text"
                ]
            ),
            sep=None,
            engine="python",
            dtype=str,
            keep_default_na=False,
        )

    except Exception as error:

        return {
            "success": False,
            "message": (
                "CSV tidak dapat dibaca: "
                f"{error}"
            ),
        }


    return {
        "success": True,
        "message": (
            "CSV berhasil dibaca."
        ),
        "dataframe": dataframe,
        "encoding": decoded[
            "encoding"
        ],
    }


# ============================================================
# 8. DETEKSI KOLOM ARAB
# ============================================================

def detect_arabic_columns(dataframe):

    results = []


    for column in dataframe.columns:

        values = dataframe[
            column
        ].astype(str)


        non_empty = 0
        arabic_rows = 0


        for value in values:

            if not value.strip():
                continue

            non_empty += 1


            if any(
                is_arabic_letter(char)
                for char in value
            ):

                arabic_rows += 1


        ratio = (
            arabic_rows / non_empty
            if non_empty > 0
            else 0.0
        )


        if arabic_rows > 0:

            results.append({
                "column": column,
                "arabic_rows":
                    arabic_rows,
                "non_empty_rows":
                    non_empty,
                "ratio":
                    ratio,
            })


    results.sort(
        key=lambda item:
            item["ratio"],
        reverse=True,
    )


    return results


# ============================================================
# 9. HELPER NAMA KOLOM BARU
# ============================================================

def safe_column_name(
    dataframe,
    base_name,
):

    if base_name not in dataframe.columns:
        return base_name

    number = 2

    while (
        f"{base_name}_{number}"
        in dataframe.columns
    ):

        number += 1


    return (
        f"{base_name}_{number}"
    )


# ============================================================
# 10. PROSES CSV
# ============================================================

def process_csv_bytes(
    data,
    filename="input.csv",
    text_column=None,
    progress_callback=None,
):

    read_result = read_csv_bytes(
        data
    )


    if not read_result[
        "success"
    ]:

        return read_result


    dataframe = (
        read_result[
            "dataframe"
        ]
        .copy()
    )


    candidates = detect_arabic_columns(
        dataframe
    )


    # --------------------------------------------------------
    # Pilih kolom otomatis jika tidak diberikan
    # --------------------------------------------------------

    if text_column is None:

        if not candidates:

            return {
                "success": False,
                "message": (
                    "Tidak ditemukan kolom yang "
                    "mengandung teks Arab."
                ),
            }

        text_column = candidates[
            0
        ][
            "column"
        ]


    if text_column not in dataframe.columns:

        return {
            "success": False,
            "message": (
                f"Kolom '{text_column}' "
                "tidak ditemukan."
            ),
        }


    # --------------------------------------------------------
    # Nama kolom output
    # --------------------------------------------------------

    output_col = safe_column_name(
        dataframe,
        "hasil_diacritization",
    )

    status_col = safe_column_name(
        dataframe,
        "status_diacritization",
    )

    note_col = safe_column_name(
        dataframe,
        "catatan_diacritization",
    )


    predictions = []
    statuses = []
    notes = []
    unknown_characters = []


    total_rows = len(
        dataframe
    )


    for row_number, value in enumerate(
        dataframe[
            text_column
        ].astype(str),
        start=1,
    ):

        result = process_text_unit(
            value
        )


        predictions.append(
            result[
                "prediction"
            ]
        )

        statuses.append(
            result[
                "status"
            ]
        )

        notes.append(
            result[
                "note"
            ]
        )


        for char in result.get(
            "unknown_characters",
            [],
        ):

            if char not in unknown_characters:

                unknown_characters.append(
                    char
                )


        if progress_callback is not None:

            progress_callback(
                row_number,
                total_rows,
            )


    dataframe[
        output_col
    ] = predictions

    dataframe[
        status_col
    ] = statuses

    dataframe[
        note_col
    ] = notes


    status_summary = dict(
        Counter(
            statuses
        )
    )


    csv_text = dataframe.to_csv(
        index=False
    )


    # UTF-8-SIG agar lebih nyaman dibuka Excel
    download_bytes = csv_text.encode(
        "utf-8-sig"
    )


    output_filename = (
        f"{Path(filename).stem}"
        "_hasil_diakritisasi.csv"
    )


    return {
        "success": True,
        "message": (
            "File CSV berhasil diproses."
        ),
        "input_type": "csv",
        "text_column": text_column,
        "detected_columns": candidates,
        "dataframe": dataframe,
        "status_summary": status_summary,
        "unknown_characters": unknown_characters,
        "output_columns": {
            "prediction":
                output_col,
            "status":
                status_col,
            "note":
                note_col,
        },
        "output_filename":
            output_filename,
        "download_bytes":
            download_bytes,
    }


# ============================================================
# 11. UNIVERSAL FILE PROCESSOR
# ============================================================

def process_uploaded_file(
    uploaded_file,
    text_column=None,
    progress_callback=None,
):

    if uploaded_file is None:

        return {
            "success": False,
            "message":
                "File belum dipilih.",
        }


    filename = getattr(
        uploaded_file,
        "name",
        "uploaded_file",
    )


    data = get_uploaded_file_bytes(
        uploaded_file
    )


    if not data:

        return {
            "success": False,
            "message":
                "File kosong.",
        }


    if len(data) > MAX_FILE_SIZE:

        return {
            "success": False,
            "message":
                "Ukuran file melebihi batas 5 MB.",
        }


    suffix = Path(
        filename
    ).suffix.lower()


    if suffix == ".txt":

        return process_txt_bytes(
            data,
            filename=filename,
            progress_callback=
                progress_callback,
        )


    if suffix == ".csv":

        return process_csv_bytes(
            data,
            filename=filename,
            text_column=text_column,
            progress_callback=
                progress_callback,
        )


    return {
        "success": False,
        "message": (
            "Format file tidak didukung. "
            "Gunakan TXT atau CSV."
        ),
    }


# ============================================================
# 12. TEST TXT
# ============================================================

txt_test = (
    "قال الشيخ الإمام العالم\n"
    "\n"
    "12345\n"
    "قَالَ الشَّيْخُ\n"
    "قال الشيخ hello\n"
)


txt_result = process_txt_bytes(
    txt_test.encode(
        "utf-8"
    ),
    filename="uji.txt",
)


print("\nTEST TXT")
print("-" * 70)

print(
    "Success :",
    txt_result[
        "success"
    ]
)

print(
    "Status  :",
    txt_result[
        "status_summary"
    ]
)

print(
    "Warning :",
    txt_result[
        "warnings"
    ]
)

print(
    "Unknown :",
    txt_result[
        "unknown_characters"
    ]
)

print("\nPreview:")
print(
    txt_result[
        "prediction"
    ]
)


assert txt_result[
    "success"
] is True

assert (
    txt_result[
        "prediction"
    ].count("\n")
    ==
    txt_test.count("\n")
)

assert (
    "campuran"
    in txt_result[
        "status_summary"
    ]
)


print(
    "✅ Pipeline TXT berhasil."
)


# ============================================================
# 13. TEST CSV
# ============================================================

csv_test = """id,teks,keterangan
1,قال الشيخ الإمام,arab
2,قَالَ الشَّيْخُ,berharakat
3,,kosong
4,12345,angka
5,قال الشيخ hello,campuran
"""


csv_read = read_csv_bytes(
    csv_test.encode(
        "utf-8"
    )
)


assert csv_read[
    "success"
] is True


detected = detect_arabic_columns(
    csv_read[
        "dataframe"
    ]
)


print("\nTEST DETEKSI KOLOM CSV")
print("-" * 70)

print(
    detected
)


assert any(
    item[
        "column"
    ] == "teks"
    for item in detected
)


csv_result = process_csv_bytes(
    csv_test.encode(
        "utf-8"
    ),
    filename="uji.csv",
    text_column="teks",
)


print("\nTEST CSV")
print("-" * 70)

print(
    "Success :",
    csv_result[
        "success"
    ]
)

print(
    "Column  :",
    csv_result[
        "text_column"
    ]
)

print(
    "Status  :",
    csv_result[
        "status_summary"
    ]
)

print(
    "Unknown :",
    csv_result[
        "unknown_characters"
    ]
)

print(
    "\nOutput columns:"
)

print(
    csv_result[
        "output_columns"
    ]
)


print("\nPreview:")
print(
    csv_result[
        "dataframe"
    ].to_string(
        index=False
    )
)


assert csv_result[
    "success"
] is True


assert (
    "hasil_diacritization"
    in csv_result[
        "dataframe"
    ].columns
)


assert (
    "status_diacritization"
    in csv_result[
        "dataframe"
    ].columns
)


assert (
    "catatan_diacritization"
    in csv_result[
        "dataframe"
    ].columns
)


print(
    "✅ Pipeline CSV berhasil."
)


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 70)

print(
    "✅ TXT dapat dibaca dan diproses."
)

print(
    "✅ Struktur baris TXT dipertahankan."
)

print(
    "✅ CSV dapat dibaca."
)

print(
    "✅ Kolom Arab dapat dideteksi."
)

print(
    "✅ Status kosong/non-Arab/campuran ditangani."
)

print(
    "✅ Output TXT/CSV siap untuk download."
)

print(
    "✅ Cell 10 selesai."
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 11 — GENERATE inference_engine.py
# ============================================================

from pathlib import Path
import py_compile


print("=" * 70)
print("GENERATE inference_engine.py")
print("=" * 70)


ENGINE_PATH = (
    STREAMLIT_DIR
    / "inference_engine.py"
)


engine_source = r'''
# ============================================================
# inference_engine.py
# Backend Sistem Diakritisasi Otomatis Teks Arab Gundul
# Model BiLSTM-CRF
# ============================================================

from pathlib import Path
from io import StringIO
from collections import Counter
import json
import os
import re
import unicodedata

import pandas as pd
import torch
import torch.nn as nn

from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
)

from torchcrf import CRF


# ============================================================
# 1. PATH PROJECT
# ============================================================

APP_DIR = (
    Path(__file__)
    .resolve()
    .parent
)

DEFAULT_PROJECT_ROOT = (
    APP_DIR.parent
)

PROJECT_ROOT = Path(
    os.environ.get(
        "BILSTM_CRF_PROJECT_ROOT",
        str(DEFAULT_PROJECT_ROOT),
    )
).resolve()


MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "finetuned"
    / "best_finetuned_model.pt"
)

CONFIG_PATH = (
    PROJECT_ROOT
    / "models"
    / "finetuned"
    / "finetuning_config.json"
)

CHAR2ID_PATH = (
    PROJECT_ROOT
    / "mappings"
    / "char2id.json"
)

LABEL2ID_PATH = (
    PROJECT_ROOT
    / "mappings"
    / "label2id.json"
)

ID2LABEL_PATH = (
    PROJECT_ROOT
    / "mappings"
    / "id2label.json"
)

METRICS_PATH = (
    PROJECT_ROOT
    / "evaluation"
    / "comparison"
    / "metrics_comparison.json"
)


MAX_FILE_SIZE = (
    5 * 1024 * 1024
)


# ============================================================
# 2. GLOBAL BACKEND STATE
# ============================================================

_DEVICE = None
_MODEL = None
_CONFIG = None
_CHAR2ID = None
_LABEL2ID = None
_ID2LABEL = None
_METRICS = None
_MODEL_INFO = None


# ============================================================
# 3. JSON HELPER
# ============================================================

def _load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as file:

        return json.load(file)


# ============================================================
# 4. MODEL
# ============================================================

class BiLSTMCRF(nn.Module):

    def __init__(
        self,
        vocab_size,
        num_labels,
        embedding_dim,
        hidden_dim,
        num_layers,
        dropout,
        pad_idx,
    ):

        super().__init__()


        if hidden_dim % 2 != 0:

            raise ValueError(
                "hidden_dim BiLSTM harus genap."
            )


        hidden_per_direction = (
            hidden_dim // 2
        )


        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_idx,
        )


        self.embedding_dropout = nn.Dropout(
            dropout
        )


        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_per_direction,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=(
                dropout
                if num_layers > 1
                else 0.0
            ),
        )


        self.output_dropout = nn.Dropout(
            dropout
        )


        self.classifier = nn.Linear(
            hidden_dim,
            num_labels,
        )


        self.crf = CRF(
            num_tags=num_labels,
            batch_first=True,
        )


    def forward(
        self,
        input_ids,
        lengths,
    ):

        embeddings = self.embedding(
            input_ids
        )


        embeddings = self.embedding_dropout(
            embeddings
        )


        packed = pack_padded_sequence(
            embeddings,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )


        packed_output, _ = self.bilstm(
            packed
        )


        bilstm_output, _ = (
            pad_packed_sequence(
                packed_output,
                batch_first=True,
                total_length=
                    input_ids.size(1),
            )
        )


        bilstm_output = self.output_dropout(
            bilstm_output
        )


        return self.classifier(
            bilstm_output
        )


    def loss(
        self,
        input_ids,
        tags,
        mask,
        lengths,
    ):

        emissions = self.forward(
            input_ids,
            lengths,
        )


        log_likelihood = self.crf(
            emissions,
            tags,
            mask=mask,
            reduction="token_mean",
        )


        return -log_likelihood


    def decode(
        self,
        input_ids,
        mask,
        lengths,
    ):

        emissions = self.forward(
            input_ids,
            lengths,
        )


        return self.crf.decode(
            emissions,
            mask=mask,
        )


# ============================================================
# 5. LOAD BACKEND
# ============================================================

def _ensure_loaded():

    global _DEVICE
    global _MODEL
    global _CONFIG
    global _CHAR2ID
    global _LABEL2ID
    global _ID2LABEL
    global _METRICS
    global _MODEL_INFO


    if _MODEL is not None:

        return


    required_files = [
        MODEL_PATH,
        CONFIG_PATH,
        CHAR2ID_PATH,
        LABEL2ID_PATH,
        ID2LABEL_PATH,
        METRICS_PATH,
    ]


    missing = [
        str(path)
        for path in required_files
        if not path.exists()
    ]


    if missing:

        raise FileNotFoundError(
            "File backend tidak ditemukan:\n"
            +
            "\n".join(
                missing
            )
        )


    _DEVICE = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


    _CONFIG = _load_json(
        CONFIG_PATH
    )

    _CHAR2ID = _load_json(
        CHAR2ID_PATH
    )

    _LABEL2ID = _load_json(
        LABEL2ID_PATH
    )


    id2label_raw = _load_json(
        ID2LABEL_PATH
    )


    _ID2LABEL = {
        int(key): value
        for key, value
        in id2label_raw.items()
    }


    _METRICS = _load_json(
        METRICS_PATH
    )


    vocab_size = len(
        _CHAR2ID
    )

    num_labels = len(
        _LABEL2ID
    )


    pad_id = int(
        _CHAR2ID[
            "<PAD>"
        ]
    )


    checkpoint = torch.load(
        MODEL_PATH,
        map_location=_DEVICE,
        weights_only=False,
    )


    if (
        not isinstance(
            checkpoint,
            dict
        )
        or
        "model_state_dict"
        not in checkpoint
    ):

        raise RuntimeError(
            "Format checkpoint tidak valid."
        )


    state_dict = checkpoint[
        "model_state_dict"
    ]


    # Bersihkan prefix DataParallel bila ada.
    state_dict = {
        (
            key[len("module."):]
            if key.startswith("module.")
            else key
        ): value

        for key, value
        in state_dict.items()
    }


    model = BiLSTMCRF(
        vocab_size=vocab_size,

        num_labels=num_labels,

        embedding_dim=int(
            _CONFIG[
                "embedding_dimension"
            ]
        ),

        hidden_dim=int(
            _CONFIG[
                "hidden_dimension"
            ]
        ),

        num_layers=int(
            _CONFIG[
                "number_of_lstm_layers"
            ]
        ),

        dropout=float(
            _CONFIG[
                "dropout"
            ]
        ),

        pad_idx=pad_id,
    )


    model.load_state_dict(
        state_dict,
        strict=True,
    )


    model = model.to(
        _DEVICE
    )

    model.eval()


    _MODEL = model


    total_parameters = sum(
        parameter.numel()
        for parameter
        in model.parameters()
    )


    hidden_dimension = int(
        _CONFIG[
            "hidden_dimension"
        ]
    )


    _MODEL_INFO = {

        "device":
            str(_DEVICE),

        "vocab_size":
            vocab_size,

        "num_labels":
            num_labels,

        "embedding_dim":
            int(
                _CONFIG[
                    "embedding_dimension"
                ]
            ),

        "hidden_dim":
            hidden_dimension,

        "hidden_per_direction":
            hidden_dimension // 2,

        "num_layers":
            int(
                _CONFIG[
                    "number_of_lstm_layers"
                ]
            ),

        "dropout":
            float(
                _CONFIG[
                    "dropout"
                ]
            ),

        "max_len":
            int(
                _CONFIG[
                    "max_sequence_length"
                ]
            ),

        "total_parameters":
            total_parameters,

        "checkpoint_epoch":
            checkpoint.get(
                "epoch"
            ),

        "model_name":
            _CONFIG.get(
                "model_name",
                "BiLSTM-CRF",
            ),
    }


# ============================================================
# 6. PREPROCESSING
# ============================================================

def is_arabic_diacritic(char):

    if not char:
        return False


    if (
        unicodedata.category(
            char
        )
        !=
        "Mn"
    ):

        return False


    return (
        "ARABIC"
        in unicodedata.name(
            char,
            ""
        )
    )


def remove_arabic_diacritics(text):

    if text is None:
        return ""


    return "".join(
        char

        for char
        in str(text)

        if not is_arabic_diacritic(
            char
        )
    )


def is_arabic_letter(char):

    if not char:
        return False


    return (
        unicodedata.category(
            char
        ).startswith("L")

        and

        "ARABIC"
        in unicodedata.name(
            char,
            ""
        )
    )


def is_latin_letter(char):

    if not char:
        return False


    return (
        unicodedata.category(
            char
        ).startswith("L")

        and

        "LATIN"
        in unicodedata.name(
            char,
            ""
        )
    )


def normalize_text(text):

    if text is None:
        return ""

    text = unicodedata.normalize(
        "NFC",
        str(text),
    )

    # Samakan normalisasi inferensi dengan prapemrosesan data pelatihan.
    text = text.replace("ـ", "")
    text = text.replace("\u0670", "")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def count_arabic_letters(text):

    return sum(
        1
        for char in text
        if is_arabic_letter(
            char
        )
    )


def find_latin_letters(text):

    return sorted(
        set(
            char

            for char
            in text

            if is_latin_letter(
                char
            )
        )
    )


def contains_arabic_diacritics(text):

    return any(
        is_arabic_diacritic(
            char
        )
        for char in text
    )


def validate_input(text):

    normalized = normalize_text(
        text
    )


    if not normalized.strip():

        return {
            "valid": False,
            "message":
                "Input masih kosong.",
            "normalized_text":
                normalized,
            "clean_text":
                "",
            "had_diacritics":
                False,
            "arabic_letters":
                0,
            "latin_letters":
                [],
        }


    latin_letters = find_latin_letters(
        normalized
    )


    if latin_letters:

        return {
            "valid": False,

            "message": (
                "Input manual mengandung "
                "huruf Latin. Masukkan teks "
                "Arab tanpa huruf Latin."
            ),

            "normalized_text":
                normalized,

            "clean_text":
                "",

            "had_diacritics":
                contains_arabic_diacritics(
                    normalized
                ),

            "arabic_letters":
                count_arabic_letters(
                    normalized
                ),

            "latin_letters":
                latin_letters,
        }


    arabic_letters = count_arabic_letters(
        normalized
    )


    if arabic_letters == 0:

        return {
            "valid": False,

            "message":
                "Tidak ditemukan huruf Arab "
                "pada input.",

            "normalized_text":
                normalized,

            "clean_text":
                "",

            "had_diacritics":
                False,

            "arabic_letters":
                0,

            "latin_letters":
                [],
        }


    had_diacritics = (
        contains_arabic_diacritics(
            normalized
        )
    )


    clean_text = (
        remove_arabic_diacritics(
            normalized
        )
    )


    clean_text = unicodedata.normalize(
        "NFC",
        clean_text,
    )


    return {
        "valid":
            True,

        "message":
            "Input valid.",

        "normalized_text":
            normalized,

        "clean_text":
            clean_text,

        "had_diacritics":
            had_diacritics,

        "arabic_letters":
            arabic_letters,

        "latin_letters":
            [],
    }


# ============================================================
# 7. CHUNKING
# ============================================================

def split_long_text(
    text,
    max_len=None,
):

    _ensure_loaded()


    if max_len is None:

        max_len = int(
            _MODEL_INFO[
                "max_len"
            ]
        )


    text = normalize_text(
        text
    )


    if not text:

        return []


    if max_len <= 0:

        raise ValueError(
            "max_len harus lebih besar dari 0."
        )


    if len(text) <= max_len:

        return [
            text
        ]


    chunks = []

    start = 0
    text_length = len(
        text
    )


    while start < text_length:

        end = min(
            start + max_len,
            text_length,
        )


        if end == text_length:

            chunk = text[
                start:end
            ]


            if chunk:

                chunks.append(
                    chunk
                )


            break


        split_position = None


        for index in range(
            end - 1,
            start,
            -1,
        ):

            if text[
                index
            ].isspace():

                split_position = (
                    index + 1
                )

                break


        if split_position is None:

            split_position = end


        chunk = text[
            start:split_position
        ]


        if chunk:

            chunks.append(
                chunk
            )


        start = split_position


    return chunks


# ============================================================
# 8. RECONSTRUCTION
# ============================================================

def label_to_diacritics(label):

    if label == "O":
        return ""


    return "".join(
        part

        for part
        in label.split("|")

        if part
    )


def reconstruct_diacritized_text(
    clean_text,
    predicted_label_ids,
):

    _ensure_loaded()


    if len(
        clean_text
    ) != len(
        predicted_label_ids
    ):

        raise ValueError(
            "Panjang teks dan label prediksi berbeda."
        )


    output = []


    for char, label_id in zip(
        clean_text,
        predicted_label_ids,
    ):

        label_id = int(
            label_id
        )


        if label_id not in _ID2LABEL:

            raise KeyError(
                f"ID label tidak dikenal: "
                f"{label_id}"
            )


        output.append(
            char
        )


        if is_arabic_letter(
            char
        ):

            diacritics = (
                label_to_diacritics(
                    _ID2LABEL[
                        label_id
                    ]
                )
            )


            if diacritics:

                output.append(
                    diacritics
                )


    return unicodedata.normalize(
        "NFC",
        "".join(
            output
        ),
    )


# ============================================================
# 9. INFERENCE SINGLE CHUNK
# ============================================================

def predict_single_chunk(text):

    _ensure_loaded()


    normalized = normalize_text(
        text
    )


    if not normalized:

        return {
            "success": False,
            "message": "Chunk kosong.",
            "input_text": "",
            "clean_text": "",
            "prediction": "",
            "predicted_label_ids": [],
            "predicted_labels": [],
            "unknown_characters": [],
            "length": 0,
        }


    clean_text = (
        remove_arabic_diacritics(
            normalized
        )
    )


    clean_text = unicodedata.normalize(
        "NFC",
        clean_text,
    )


    if not clean_text:

        return {
            "success": False,
            "message":
                "Tidak ada karakter yang "
                "dapat diproses.",
            "input_text":
                normalized,
            "clean_text":
                "",
            "prediction":
                "",
            "predicted_label_ids":
                [],
            "predicted_labels":
                [],
            "unknown_characters":
                [],
            "length":
                0,
        }


    max_len = int(
        _MODEL_INFO[
            "max_len"
        ]
    )


    if len(
        clean_text
    ) > max_len:

        raise ValueError(
            f"Chunk terlalu panjang: "
            f"{len(clean_text)} karakter. "
            f"Maksimum {max_len}."
        )


    unk_id = int(
        _CHAR2ID[
            "<UNK>"
        ]
    )


    input_ids = []
    unknown_characters = []


    for char in clean_text:

        if char in _CHAR2ID:

            char_id = _CHAR2ID[
                char
            ]

        else:

            char_id = unk_id


            if (
                char
                not in
                unknown_characters
            ):

                unknown_characters.append(
                    char
                )


        input_ids.append(
            char_id
        )


    input_tensor = torch.tensor(
        [
            input_ids
        ],
        dtype=torch.long,
        device=_DEVICE,
    )


    lengths = torch.tensor(
        [
            len(
                input_ids
            )
        ],
        dtype=torch.long,
    )


    mask = torch.ones(
        (
            1,
            len(
                input_ids
            ),
        ),
        dtype=torch.bool,
        device=_DEVICE,
    )


    _MODEL.eval()


    with torch.inference_mode():

        decoded = _MODEL.decode(
            input_tensor,
            mask,
            lengths,
        )


    if len(
        decoded
    ) != 1:

        raise RuntimeError(
            "CRF menghasilkan jumlah "
            "sequence tidak sesuai."
        )


    predicted_label_ids = (
        decoded[
            0
        ]
    )


    if len(
        predicted_label_ids
    ) != len(
        clean_text
    ):

        raise RuntimeError(
            "Jumlah label tidak sama "
            "dengan panjang input."
        )


    predicted_labels = [
        _ID2LABEL[
            int(label_id)
        ]

        for label_id
        in predicted_label_ids
    ]


    prediction = (
        reconstruct_diacritized_text(
            clean_text,
            predicted_label_ids,
        )
    )


    return {
        "success":
            True,

        "message":
            "Inference berhasil.",

        "input_text":
            normalized,

        "clean_text":
            clean_text,

        "prediction":
            prediction,

        "predicted_label_ids":
            predicted_label_ids,

        "predicted_labels":
            predicted_labels,

        "unknown_characters":
            unknown_characters,

        "length":
            len(
                clean_text
            ),
    }


# ============================================================
# 10. INFERENCE LONG TEXT
# ============================================================

def predict_long_text(
    text,
    progress_callback=None,
):

    _ensure_loaded()


    validation = validate_input(
        text
    )


    if not validation[
        "valid"
    ]:

        return {
            "success":
                False,

            "message":
                validation[
                    "message"
                ],

            "input_text":
                validation[
                    "normalized_text"
                ],

            "clean_text":
                validation[
                    "clean_text"
                ],

            "prediction":
                "",

            "had_diacritics":
                validation[
                    "had_diacritics"
                ],

            "arabic_letters":
                validation[
                    "arabic_letters"
                ],

            "total_characters":
                0,

            "total_chunks":
                0,

            "chunk_lengths":
                [],

            "unknown_characters":
                [],
        }


    clean_text = validation[
        "clean_text"
    ]


    chunks = split_long_text(
        clean_text,
        max_len=int(
            _MODEL_INFO[
                "max_len"
            ]
        ),
    )


    if not chunks:

        return {
            "success":
                False,

            "message":
                "Tidak ada teks yang "
                "dapat diproses.",

            "input_text":
                validation[
                    "normalized_text"
                ],

            "clean_text":
                clean_text,

            "prediction":
                "",

            "had_diacritics":
                validation[
                    "had_diacritics"
                ],

            "arabic_letters":
                validation[
                    "arabic_letters"
                ],

            "total_characters":
                len(
                    clean_text
                ),

            "total_chunks":
                0,

            "chunk_lengths":
                [],

            "unknown_characters":
                [],
        }


    predictions = []
    unknown_characters = []

    total_chunks = len(
        chunks
    )


    for index, chunk in enumerate(
        chunks,
        start=1,
    ):

        result = predict_single_chunk(
            chunk
        )


        if not result[
            "success"
        ]:

            return {
                "success":
                    False,

                "message": (
                    f"Gagal memproses chunk "
                    f"{index}/{total_chunks}: "
                    f"{result['message']}"
                ),

                "input_text":
                    validation[
                        "normalized_text"
                    ],

                "clean_text":
                    clean_text,

                "prediction":
                    "",

                "had_diacritics":
                    validation[
                        "had_diacritics"
                    ],

                "arabic_letters":
                    validation[
                        "arabic_letters"
                    ],

                "total_characters":
                    len(
                        clean_text
                    ),

                "total_chunks":
                    total_chunks,

                "chunk_lengths": [
                    len(
                        item
                    )
                    for item in chunks
                ],

                "unknown_characters":
                    unknown_characters,
            }


        predictions.append(
            result[
                "prediction"
            ]
        )


        for char in result[
            "unknown_characters"
        ]:

            if (
                char
                not in
                unknown_characters
            ):

                unknown_characters.append(
                    char
                )


        if progress_callback is not None:

            progress_callback(
                index,
                total_chunks,
            )


    prediction = unicodedata.normalize(
        "NFC",
        "".join(
            predictions
        ),
    )


    reconstructed_clean = (
        remove_arabic_diacritics(
            prediction
        )
    )


    if (
        reconstructed_clean
        !=
        clean_text
    ):

        raise RuntimeError(
            "Rekonstruksi hasil inference "
            "tidak sama dengan input bersih."
        )


    return {
        "success":
            True,

        "message":
            "Diakritisasi berhasil.",

        "input_text":
            validation[
                "normalized_text"
            ],

        "clean_text":
            clean_text,

        "prediction":
            prediction,

        "had_diacritics":
            validation[
                "had_diacritics"
            ],

        "arabic_letters":
            validation[
                "arabic_letters"
            ],

        "total_characters":
            len(
                clean_text
            ),

        "total_chunks":
            total_chunks,

        "chunk_lengths": [
            len(
                chunk
            )
            for chunk in chunks
        ],

        "unknown_characters":
            unknown_characters,
    }


# ============================================================
# 11. FILE UTILITIES
# ============================================================

def get_uploaded_file_bytes(uploaded_file):

    if uploaded_file is None:
        return b""


    if isinstance(
        uploaded_file,
        bytes,
    ):

        return uploaded_file


    if isinstance(
        uploaded_file,
        bytearray,
    ):

        return bytes(
            uploaded_file
        )


    if hasattr(
        uploaded_file,
        "getvalue",
    ):

        return uploaded_file.getvalue()


    if hasattr(
        uploaded_file,
        "read",
    ):

        position = None


        if hasattr(
            uploaded_file,
            "tell",
        ):

            try:
                position = uploaded_file.tell()
            except Exception:
                pass


        data = uploaded_file.read()


        if (
            position is not None
            and
            hasattr(
                uploaded_file,
                "seek",
            )
        ):

            try:
                uploaded_file.seek(
                    position
                )
            except Exception:
                pass


        return data


    raise TypeError(
        "Format file tidak didukung."
    )


def decode_text_bytes(data):

    for encoding in [
        "utf-8-sig",
        "utf-8",
        "utf-16",
        "cp1256",
    ]:

        try:

            return {
                "success":
                    True,

                "text":
                    data.decode(
                        encoding
                    ),

                "encoding":
                    encoding,

                "message": (
                    f"File berhasil dibaca "
                    f"dengan encoding "
                    f"{encoding}."
                ),
            }

        except UnicodeDecodeError:
            continue


    return {
        "success":
            False,

        "text":
            "",

        "encoding":
            None,

        "message": (
            "Encoding file tidak dikenali. "
            "Gunakan UTF-8 jika memungkinkan."
        ),
    }


def classify_text_unit(text):

    text = normalize_text(
        text
    )


    if not text.strip():
        return "kosong"


    has_arabic = any(
        is_arabic_letter(
            char
        )
        for char in text
    )


    has_latin = any(
        is_latin_letter(
            char
        )
        for char in text
    )


    if has_arabic and has_latin:
        return "campuran"


    if has_arabic:
        return "arab"


    return "non_arab"


def process_text_unit(text):

    normalized = normalize_text(
        text
    )


    category = classify_text_unit(
        normalized
    )


    if category == "kosong":

        return {
            "success":
                True,
            "status":
                "kosong",
            "input":
                normalized,
            "prediction":
                normalized,
            "note":
                "Teks kosong dipertahankan.",
            "unknown_characters":
                [],
        }


    if category == "non_arab":

        return {
            "success":
                True,
            "status":
                "non_arab",
            "input":
                normalized,
            "prediction":
                normalized,
            "note": (
                "Tidak mengandung huruf Arab; "
                "teks dipertahankan."
            ),
            "unknown_characters":
                [],
        }


    if category == "campuran":

        return {
            "success":
                True,
            "status":
                "campuran",
            "input":
                normalized,
            "prediction":
                normalized,
            "note": (
                "Teks mengandung campuran "
                "huruf Arab dan Latin; "
                "dipertahankan tanpa prediksi."
            ),
            "unknown_characters":
                [],
        }


    result = predict_long_text(
        normalized
    )


    if not result[
        "success"
    ]:

        return {
            "success":
                False,
            "status":
                "gagal",
            "input":
                normalized,
            "prediction":
                normalized,
            "note":
                result[
                    "message"
                ],
            "unknown_characters":
                [],
        }


    note = (
        "Diakritisasi berhasil."
    )


    if result[
        "had_diacritics"
    ]:

        note = (
            "Harakat lama dihapus "
            "dan teks diprediksi ulang."
        )


    return {
        "success":
            True,

        "status":
            "berhasil",

        "input":
            normalized,

        "prediction":
            result[
                "prediction"
            ],

        "note":
            note,

        "unknown_characters":
            result[
                "unknown_characters"
            ],
    }


def split_line_ending(line):

    if line.endswith(
        "\r\n"
    ):
        return (
            line[:-2],
            "\r\n",
        )


    if line.endswith(
        "\n"
    ):
        return (
            line[:-1],
            "\n",
        )


    if line.endswith(
        "\r"
    ):
        return (
            line[:-1],
            "\r",
        )


    return (
        line,
        "",
    )


# ============================================================
# 12. TXT
# ============================================================

def process_txt_bytes(
    data,
    filename="input.txt",
    progress_callback=None,
):

    decoded = decode_text_bytes(
        data
    )


    if not decoded[
        "success"
    ]:

        return {
            "success":
                False,

            "message":
                decoded[
                    "message"
                ],
        }


    text = decoded[
        "text"
    ]


    lines = text.splitlines(
        keepends=True
    )


    if (
        not lines
        and
        text
    ):

        lines = [
            text
        ]


    output_parts = []
    statuses = []
    warnings = []
    unknown_characters = []

    total_lines = len(
        lines
    )


    for index, raw_line in enumerate(
        lines,
        start=1,
    ):

        content, line_ending = (
            split_line_ending(
                raw_line
            )
        )


        result = process_text_unit(
            content
        )


        output_parts.append(
            result[
                "prediction"
            ]
            +
            line_ending
        )


        statuses.append(
            result[
                "status"
            ]
        )


        for char in result.get(
            "unknown_characters",
            [],
        ):

            if (
                char
                not in
                unknown_characters
            ):

                unknown_characters.append(
                    char
                )


        if (
            result[
                "status"
            ]
            ==
            "campuran"
        ):

            warnings.append({
                "line":
                    index,

                "note":
                    result[
                        "note"
                    ],
            })


        if progress_callback is not None:

            progress_callback(
                index,
                total_lines,
            )


    prediction = "".join(
        output_parts
    )


    output_filename = (
        f"{Path(filename).stem}"
        "_hasil_diakritisasi.txt"
    )


    return {
        "success":
            True,

        "message":
            "File TXT berhasil diproses.",

        "input_type":
            "txt",

        "encoding":
            decoded[
                "encoding"
            ],

        "prediction":
            prediction,

        "status_summary":
            dict(
                Counter(
                    statuses
                )
            ),

        "warnings":
            warnings,

        "unknown_characters":
            unknown_characters,

        "output_filename":
            output_filename,

        "download_bytes":
            prediction.encode(
                "utf-8"
            ),
    }


# ============================================================
# 13. CSV
# ============================================================

def read_csv_bytes(data):

    decoded = decode_text_bytes(
        data
    )


    if not decoded[
        "success"
    ]:

        return {
            "success":
                False,

            "message":
                decoded[
                    "message"
                ],
        }


    try:

        dataframe = pd.read_csv(
            StringIO(
                decoded[
                    "text"
                ]
            ),

            sep=None,
            engine="python",
            dtype=str,
            keep_default_na=False,
        )

    except Exception as error:

        return {
            "success":
                False,

            "message": (
                "CSV tidak dapat dibaca: "
                f"{error}"
            ),
        }


    return {
        "success":
            True,

        "message":
            "CSV berhasil dibaca.",

        "dataframe":
            dataframe,

        "encoding":
            decoded[
                "encoding"
            ],
    }


def read_csv_file(uploaded_file):

    data = get_uploaded_file_bytes(
        uploaded_file
    )


    if not data:

        return {
            "success":
                False,

            "message":
                "File kosong.",
        }


    if len(
        data
    ) > MAX_FILE_SIZE:

        return {
            "success":
                False,

            "message":
                "Ukuran file melebihi 5 MB.",
        }


    return read_csv_bytes(
        data
    )


def detect_arabic_columns(dataframe):

    results = []


    for column in dataframe.columns:

        values = dataframe[
            column
        ].astype(
            str
        )


        non_empty = 0
        arabic_rows = 0


        for value in values:

            if not value.strip():
                continue


            non_empty += 1


            if any(
                is_arabic_letter(
                    char
                )
                for char in value
            ):

                arabic_rows += 1


        ratio = (
            arabic_rows / non_empty
            if non_empty > 0
            else 0.0
        )


        if arabic_rows > 0:

            results.append({
                "column":
                    column,

                "arabic_rows":
                    arabic_rows,

                "non_empty_rows":
                    non_empty,

                "ratio":
                    ratio,
            })


    results.sort(
        key=lambda item:
            item[
                "ratio"
            ],

        reverse=True,
    )


    return results


def safe_column_name(
    dataframe,
    base_name,
):

    if (
        base_name
        not in
        dataframe.columns
    ):

        return base_name


    number = 2


    while (
        f"{base_name}_{number}"
        in dataframe.columns
    ):

        number += 1


    return (
        f"{base_name}_{number}"
    )


def process_csv_bytes(
    data,
    filename="input.csv",
    text_column=None,
    progress_callback=None,
):

    read_result = read_csv_bytes(
        data
    )


    if not read_result[
        "success"
    ]:

        return read_result


    dataframe = (
        read_result[
            "dataframe"
        ]
        .copy()
    )


    candidates = detect_arabic_columns(
        dataframe
    )


    if text_column is None:

        if not candidates:

            return {
                "success":
                    False,

                "message": (
                    "Tidak ditemukan kolom "
                    "yang mengandung teks Arab."
                ),
            }


        text_column = (
            candidates[
                0
            ][
                "column"
            ]
        )


    if (
        text_column
        not in
        dataframe.columns
    ):

        return {
            "success":
                False,

            "message": (
                f"Kolom '{text_column}' "
                "tidak ditemukan."
            ),
        }


    output_col = safe_column_name(
        dataframe,
        "hasil_diacritization",
    )

    status_col = safe_column_name(
        dataframe,
        "status_diacritization",
    )

    note_col = safe_column_name(
        dataframe,
        "catatan_diacritization",
    )


    predictions = []
    statuses = []
    notes = []
    unknown_characters = []


    total_rows = len(
        dataframe
    )


    for row_number, value in enumerate(
        dataframe[
            text_column
        ].astype(
            str
        ),
        start=1,
    ):

        result = process_text_unit(
            value
        )


        predictions.append(
            result[
                "prediction"
            ]
        )

        statuses.append(
            result[
                "status"
            ]
        )

        notes.append(
            result[
                "note"
            ]
        )


        for char in result.get(
            "unknown_characters",
            [],
        ):

            if (
                char
                not in
                unknown_characters
            ):

                unknown_characters.append(
                    char
                )


        if progress_callback is not None:

            progress_callback(
                row_number,
                total_rows,
            )


    dataframe[
        output_col
    ] = predictions

    dataframe[
        status_col
    ] = statuses

    dataframe[
        note_col
    ] = notes


    csv_text = dataframe.to_csv(
        index=False
    )


    return {
        "success":
            True,

        "message":
            "File CSV berhasil diproses.",

        "input_type":
            "csv",

        "text_column":
            text_column,

        "detected_columns":
            candidates,

        "dataframe":
            dataframe,

        "status_summary":
            dict(
                Counter(
                    statuses
                )
            ),

        "unknown_characters":
            unknown_characters,

        "output_columns": {
            "prediction":
                output_col,

            "status":
                status_col,

            "note":
                note_col,
        },

        "output_filename": (
            f"{Path(filename).stem}"
            "_hasil_diakritisasi.csv"
        ),

        "download_bytes":
            csv_text.encode(
                "utf-8-sig"
            ),
    }


# ============================================================
# 14. UNIVERSAL UPLOAD PROCESSOR
# ============================================================

def process_uploaded_file(
    uploaded_file,
    text_column=None,
    progress_callback=None,
):

    if uploaded_file is None:

        return {
            "success":
                False,

            "message":
                "File belum dipilih.",
        }


    filename = getattr(
        uploaded_file,
        "name",
        "uploaded_file",
    )


    data = get_uploaded_file_bytes(
        uploaded_file
    )


    if not data:

        return {
            "success":
                False,

            "message":
                "File kosong.",
        }


    if len(
        data
    ) > MAX_FILE_SIZE:

        return {
            "success":
                False,

            "message":
                "Ukuran file melebihi batas 5 MB.",
        }


    suffix = Path(
        filename
    ).suffix.lower()


    if suffix == ".txt":

        return process_txt_bytes(
            data,

            filename=
                filename,

            progress_callback=
                progress_callback,
        )


    if suffix == ".csv":

        return process_csv_bytes(
            data,

            filename=
                filename,

            text_column=
                text_column,

            progress_callback=
                progress_callback,
        )


    return {
        "success":
            False,

        "message": (
            "Format file tidak didukung. "
            "Gunakan TXT atau CSV."
        ),
    }


# ============================================================
# 15. PUBLIC BACKEND INFO
# ============================================================

def health_check():

    try:

        _ensure_loaded()


        test_text = (
            "قال الشيخ الإمام العالم"
        )


        result = predict_single_chunk(
            test_text
        )


        if not result[
            "success"
        ]:

            raise RuntimeError(
                result[
                    "message"
                ]
            )


        return {
            "success":
                True,

            "message":
                "Backend siap digunakan.",

            "project_root":
                str(
                    PROJECT_ROOT
                ),

            "model_path":
                str(
                    MODEL_PATH
                ),

            "model":
                dict(
                    _MODEL_INFO
                ),

            "metrics":
                _METRICS,
        }


    except Exception as error:

        return {
            "success":
                False,

            "message":
                str(
                    error
                ),

            "project_root":
                str(
                    PROJECT_ROOT
                ),

            "model_path":
                str(
                    MODEL_PATH
                ),

            "model":
                None,

            "metrics":
                None,
        }


def get_model_info():

    _ensure_loaded()

    return dict(
        _MODEL_INFO
    )


def get_metrics():

    _ensure_loaded()

    return _METRICS
'''


# ------------------------------------------------------------
# 1. Tulis file
# ------------------------------------------------------------

ENGINE_PATH.write_text(
    engine_source.strip() + "\n",
    encoding="utf-8",
)


print(
    "✅ File dibuat:"
)

print(
    ENGINE_PATH
)


# ------------------------------------------------------------
# 2. Ukuran file
# ------------------------------------------------------------

file_size = (
    ENGINE_PATH
    .stat()
    .st_size
)


print(
    "\nUkuran file:",
    f"{file_size:,}",
    "bytes"
)


# ------------------------------------------------------------
# 3. Syntax validation
# ------------------------------------------------------------

py_compile.compile(
    str(
        ENGINE_PATH
    ),
    doraise=True,
)


print(
    "✅ Syntax Python valid."
)


# ------------------------------------------------------------
# 4. Validasi teks penting
# ------------------------------------------------------------

saved_source = (
    ENGINE_PATH
    .read_text(
        encoding="utf-8"
    )
)


required_markers = [

    "class BiLSTMCRF",

    "def validate_input",

    "def split_long_text",

    "def predict_single_chunk",

    "def predict_long_text",

    "def process_txt_bytes",

    "def process_csv_bytes",

    "def read_csv_file",

    "def process_uploaded_file",

    "def health_check",

    "BILSTM_CRF_PROJECT_ROOT",

    "APP_DIR.parent",
]


for marker in required_markers:

    assert marker in saved_source, (
        f"❌ Komponen hilang: {marker}"
    )


print(
    "✅ Semua fungsi backend utama tersedia."
)


# ------------------------------------------------------------
# 5. Pastikan tidak ada hardcode mount lama
# ------------------------------------------------------------

assert (
    "/content/drive/MyDrive"
    not in saved_source
), (
    "❌ Masih ada hardcoded /content/drive."
)


assert (
    "/content/gdrive/MyDrive"
    not in saved_source
), (
    "❌ Backend tidak boleh tergantung "
    "pada mount Colab tertentu."
)


print(
    "✅ Tidak ada hardcoded path Colab."
)

print(
    "✅ PROJECT_ROOT menggunakan lokasi file "
    "atau environment variable."
)


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 70)

print(
    "✅ inference_engine.py berhasil dibuat."
)

print(
    "✅ Syntax valid."
)

print(
    "✅ Path backend portable."
)

print(
    "✅ Model + preprocessing + inference tersedia."
)

print(
    "✅ TXT + CSV pipeline tersedia."
)

print(
    "✅ health_check tersedia."
)

print(
    "✅ Cell 11 selesai."
)

print("=" * 70)

In [ ]:
# ============================================================
# CELL 12 — FRESH IMPORT TEST inference_engine.py
# ============================================================

import os
import sys
import subprocess
import textwrap
from pathlib import Path


print("=" * 70)
print("FRESH IMPORT TEST BACKEND")
print("=" * 70)


# ------------------------------------------------------------
# 1. Validasi file
# ------------------------------------------------------------

ENGINE_PATH = (
    STREAMLIT_DIR
    / "inference_engine.py"
)

assert ENGINE_PATH.exists(), (
    "❌ inference_engine.py tidak ditemukan."
)

print(
    "Engine:",
    ENGINE_PATH
)


# ------------------------------------------------------------
# 2. Buat script test terpisah
#
# Script dijalankan dengan Python process BARU.
# Jadi tidak menggunakan fungsi/model yang masih ada
# di memory notebook.
# ------------------------------------------------------------

TEST_SCRIPT_PATH = Path(
    "/content/test_backend_fresh.py"
)


test_source = r'''
import os
import sys
from pathlib import Path


# ------------------------------------------------------------
# Path dari environment
# ------------------------------------------------------------

project_root = Path(
    os.environ[
        "BILSTM_CRF_PROJECT_ROOT"
    ]
)

streamlit_dir = (
    project_root
    / "streamlit_app"
)

sys.path.insert(
    0,
    str(streamlit_dir)
)


# ------------------------------------------------------------
# Import engine dari file sebenarnya
# ------------------------------------------------------------

import inference_engine as engine


print("=" * 70)
print("1. HEALTH CHECK")
print("=" * 70)


health = engine.health_check()


print(
    "Success:",
    health["success"]
)

print(
    "Message:",
    health["message"]
)

print(
    "Project:",
    health["project_root"]
)

print(
    "Model:",
    health["model_path"]
)


assert health[
    "success"
] is True, (
    health["message"]
)


# ------------------------------------------------------------
# Model info
# ------------------------------------------------------------

model_info = health[
    "model"
]


print("\nMODEL INFO")
print("-" * 70)

print(
    "Device               :",
    model_info["device"]
)

print(
    "Vocabulary           :",
    model_info["vocab_size"]
)

print(
    "Labels               :",
    model_info["num_labels"]
)

print(
    "Embedding            :",
    model_info["embedding_dim"]
)

print(
    "Hidden per direction :",
    model_info[
        "hidden_per_direction"
    ]
)

print(
    "BiLSTM output        :",
    model_info["hidden_dim"]
)

print(
    "Max sequence         :",
    model_info["max_len"]
)

print(
    "Total parameters     :",
    f"{model_info['total_parameters']:,}"
)

print(
    "Checkpoint epoch     :",
    model_info[
        "checkpoint_epoch"
    ]
)


assert (
    model_info["vocab_size"]
    == 55
)

assert (
    model_info["num_labels"]
    == 15
)

assert (
    model_info["embedding_dim"]
    == 128
)

assert (
    model_info[
        "hidden_per_direction"
    ]
    == 128
)

assert (
    model_info["hidden_dim"]
    == 256
)

assert (
    model_info["max_len"]
    == 300
)

assert (
    model_info[
        "total_parameters"
    ]
    == 275342
)

assert (
    int(
        model_info[
            "checkpoint_epoch"
        ]
    )
    == 46
)


print(
    "✅ Model info valid."
)


# ------------------------------------------------------------
# 2. Single inference
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. SINGLE INFERENCE")
print("=" * 70)


text = (
    "قال الشيخ الإمام العالم العلامة شمس الدين"
)


result = engine.predict_single_chunk(
    text
)


print(
    "Success:",
    result["success"]
)

print(
    "Input  :",
    text
)

print(
    "Output :",
    result["prediction"]
)

print(
    "Length :",
    result["length"]
)

print(
    "Labels :",
    len(
        result[
            "predicted_labels"
        ]
    )
)

print(
    "Unknown:",
    result[
        "unknown_characters"
    ]
)


assert result[
    "success"
] is True

assert result[
    "length"
] == len(
    text
)

assert len(
    result[
        "predicted_labels"
    ]
) == len(
    text
)

assert (
    result[
        "unknown_characters"
    ]
    == []
)


print(
    "✅ Single inference valid."
)


# ------------------------------------------------------------
# 3. Long inference
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. LONG INFERENCE")
print("=" * 70)


long_text = (
    (
        "قال الشيخ الإمام العالم العلامة شمس الدين "
        "رحمه الله تعالى ونفعنا بعلومه في الدارين. "
    )
    * 15
).strip()


long_result = engine.predict_long_text(
    long_text
)


print(
    "Success :",
    long_result[
        "success"
    ]
)

print(
    "Chars   :",
    long_result[
        "total_characters"
    ]
)

print(
    "Chunks  :",
    long_result[
        "total_chunks"
    ]
)

print(
    "Lengths :",
    long_result[
        "chunk_lengths"
    ]
)

print(
    "Unknown :",
    long_result[
        "unknown_characters"
    ]
)


assert long_result[
    "success"
] is True

assert long_result[
    "total_chunks"
] > 1

assert all(
    length <= 300
    for length
    in long_result[
        "chunk_lengths"
    ]
)

assert (
    engine.remove_arabic_diacritics(
        long_result[
            "prediction"
        ]
    )
    ==
    long_result[
        "clean_text"
    ]
)


print(
    "✅ Long inference valid."
)


# ------------------------------------------------------------
# 4. TXT pipeline
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. TXT PIPELINE")
print("=" * 70)


txt_data = (
    "قال الشيخ الإمام\n"
    "\n"
    "12345\n"
    "قَالَ الشَّيْخُ\n"
    "قال الشيخ hello\n"
).encode(
    "utf-8"
)


txt_result = engine.process_txt_bytes(
    txt_data,
    filename="fresh_test.txt",
)


print(
    "Success:",
    txt_result[
        "success"
    ]
)

print(
    "Status :",
    txt_result[
        "status_summary"
    ]
)

print(
    "Warning:",
    txt_result[
        "warnings"
    ]
)


assert txt_result[
    "success"
] is True

assert (
    "berhasil"
    in txt_result[
        "status_summary"
    ]
)

assert (
    "campuran"
    in txt_result[
        "status_summary"
    ]
)


print(
    "✅ TXT pipeline valid."
)


# ------------------------------------------------------------
# 5. CSV pipeline
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. CSV PIPELINE")
print("=" * 70)


csv_data = """id,teks
1,قال الشيخ الإمام
2,قَالَ الشَّيْخُ
3,
4,12345
5,قال الشيخ hello
""".encode(
    "utf-8"
)


csv_result = engine.process_csv_bytes(
    csv_data,
    filename="fresh_test.csv",
    text_column="teks",
)


print(
    "Success:",
    csv_result[
        "success"
    ]
)

print(
    "Status :",
    csv_result[
        "status_summary"
    ]
)

print(
    "Columns:",
    csv_result[
        "output_columns"
    ]
)


assert csv_result[
    "success"
] is True

assert (
    "hasil_diacritization"
    in csv_result[
        "dataframe"
    ].columns
)

assert (
    "status_diacritization"
    in csv_result[
        "dataframe"
    ].columns
)

assert (
    "catatan_diacritization"
    in csv_result[
        "dataframe"
    ].columns
)


print(
    "✅ CSV pipeline valid."
)


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 70)

print(
    "✅ Fresh import berhasil."
)

print(
    "✅ Model berhasil dimuat dari checkpoint."
)

print(
    "✅ Single inference berhasil."
)

print(
    "✅ Long inference berhasil."
)

print(
    "✅ TXT pipeline berhasil."
)

print(
    "✅ CSV pipeline berhasil."
)

print(
    "✅ Backend mandiri dan siap dipakai Streamlit."
)

print("=" * 70)
'''


TEST_SCRIPT_PATH.write_text(
    test_source.strip() + "\n",
    encoding="utf-8",
)


print(
    "✅ Fresh test script dibuat."
)


# ------------------------------------------------------------
# 3. Environment fresh process
# ------------------------------------------------------------

test_env = os.environ.copy()

test_env[
    "BILSTM_CRF_PROJECT_ROOT"
] = str(
    PROJECT_ROOT
)


# ------------------------------------------------------------
# 4. Jalankan Python process baru
# ------------------------------------------------------------

process = subprocess.run(
    [
        sys.executable,
        str(
            TEST_SCRIPT_PATH
        ),
    ],

    env=test_env,

    capture_output=True,

    text=True,

    timeout=180,
)


# ------------------------------------------------------------
# 5. Tampilkan output
# ------------------------------------------------------------

print("\nOUTPUT FRESH PROCESS")
print("-" * 70)

print(
    process.stdout
)


if process.stderr.strip():

    print("\nSTDERR")
    print("-" * 70)

    print(
        process.stderr
    )


# ------------------------------------------------------------
# 6. Validasi exit code
# ------------------------------------------------------------

assert process.returncode == 0, (
    "❌ Fresh import backend gagal. "
    "Lihat output/error di atas."
)


print("=" * 70)
print("✅ CELL 12 LULUS.")
print("✅ inference_engine.py FINAL.")
print("✅ Backend siap untuk notebook UI.")
print("=" * 70)